# Transfer of ageing models to sagittal brain and hippocampus datasets

This notebook evaluates whether representations learned from the coronal MERFISH ageing-brain dataset transfer across anatomical regions and spatial-transcriptomics platforms. It supports the external-validation analysis in **Supplementary Fig. 3** and includes additional sagittal MI-29 analyses related to the local-ageing results in **Supplementary Fig. 2c**.

## Workflow

1. Load the processed coronal training data and the trained SpiderNet checkpoint.
2. Load or reconstruct cell-type-specific age-prediction regressors and the coronal reference transformations for NMF-LR, COMMOT, and Banksy.
3. Load preprocessed sagittal MERFISH and hippocampus Stereo-seq bundles, rebuild their LR coexpression features in the coronal LR-pair space, and infer MIs with the trained SpiderNet model.
4. Predict cell-level age and summarize old-to-young mean predicted-age ratios by cell type for SpiderNet and the available baselines.
5. In the sagittal dataset, examine T-cell-sent MI-29 at the edge level and compare ageing-module scores between MI-29-high and MI-29-low T cells and their neighbors.

## Major inputs and outputs

Inputs are the processed coronal, sagittal, and hippocampus bundles; the trained coronal checkpoint; cached age regressors and baseline transformations; Banksy matrices; COMMOT results where available; and the MERFISH gene-panel table. Tables and PDF figures are written to the corresponding sagittal and hippocampus SpiderNet run directories, with a combined transfer summary written to the coronal run directory.


## 0. Dataset setup (edit this cell first)

Set the coronal training run, the processed-data bundles produced by the transfer preprocessing notebook, and the sagittal and hippocampus output roots here. The default paths below are workstation-specific and must be adapted for another environment.

In [ ]:
from workflow_paths import DATA_DIR, RESULTS_ROOT, input_path, output_path
from pathlib import Path

# =========================================================
# Training run used as the source model
# =========================================================
TRAIN_DATA_ROOT = DATA_DIR
TRAIN_OUTPUT_ROOT = RESULTS_ROOT
TRAIN_PROCESSED_DATA_DIR = (RESULTS_ROOT / 'ProcessedData')

# =========================================================
# Transfer datasets
# =========================================================
SAGITTAL_DATA_ROOT = (DATA_DIR / 'AgingBrain_Sagittal')
SAGITTAL_OUTPUT_ROOT = (RESULTS_ROOT / 'AgingBrain_Sagittal')
SAGITTAL_PROCESSED_DATA_DIR = (RESULTS_ROOT / 'AgingBrain_Sagittal/ProcessedData')

HIPPOCAMPUS_DATA_ROOT = (DATA_DIR / 'AgingBrain_Hippocampus')
HIPPOCAMPUS_OUTPUT_ROOT = (RESULTS_ROOT / 'AgingBrain_Hippocampus')
HIPPOCAMPUS_PROCESSED_DATA_DIR = (RESULTS_ROOT / 'AgingBrain_Hippocampus/ProcessedData')

# =========================================================
# Baseline feature locations
# COMMOT is available for Coronal + Hippocampus, but not Sagittal.
# Banksy is available for Coronal + Sagittal + Hippocampus.
# =========================================================
BANKSY_SUBDIR = "Banksy"
COMMOT_SUBDIR = "COMMOT"
BANKSY_LAMBDA_CELLIDENTITY = 0.2

DATASET_FEATURE_CONFIG = {
    "Coronal": {
        "output_root": TRAIN_OUTPUT_ROOT,
        "filename_hint": "coronal",
        "has_banksy": True,
        "has_commot": True,
    },
    "Sagittal": {
        "output_root": SAGITTAL_OUTPUT_ROOT,
        "filename_hint": "sagittal",
        "has_banksy": True,
        "has_commot": False,
    },
    "Hippocampus": {
        "output_root": HIPPOCAMPUS_OUTPUT_ROOT,
        "filename_hint": "hippocampus",
        "has_banksy": True,
        "has_commot": True,
    },
}

PREFERRED_METHOD_ORDER = ["SpiderNet", "NMF-LR", "COMMOT", "Banksy"]
METHOD_COLOR_DICT = {
    "SpiderNet": "#e72625",
    "NMF-LR": "#82CCE2",
    "COMMOT": "#519384",
    "Banksy": "#8A5A44",
}

# =========================================================
# Shared dataset fields
# =========================================================
SPECIES = "mouse"
SAMPLE_ID_COL = "age"
CELL_TYPE_COL = "celltype"
SPATIAL_KEY = "spatial"

# =========================================================
# Preprocessing settings
# These should match the preprocessing notebook unless you have a
# strong reason to change them.
# =========================================================
N_HVG = 1000
N_HVG_LR = 2000
NUM_NEIGHBORS = 5

# These are kept only for compatibility with helper code below.
CC_PROP_THRESHOLD = 0.01
IF_SUBSET_LRPAIR = False
NUM_SUBSET_LRPAIR_RATIO = 0.80
IF_BOTH_DIRECTIONS = False

# =========================================================
# Model / training settings
# These should match the trained SpiderNet run.
# =========================================================
DIM_ENVIR = 30
N_JOBS = 5
MAX_EPOCH = 50000
VERSION = "V1"
HIDDEN_CHANNELS = 256

# Optional manual checkpoint override.
# Set to None to automatically use the latest model_epoch*.pth.
MODEL_CHECKPOINT = None

# =========================================================
# Transfer analysis settings
# =========================================================
REPROCESS_TRANSFER_DATA = False
REFIT_REGRESSORS_IF_MISSING = True
PCA_N_COMPONENTS = 20  # gene-expression PCA baseline is disabled; this is only used for Banksy PCA features
NMF_LR_N_COMPONENTS = DIM_ENVIR
NMF_LR_RANDOM_STATE = 0
NMF_LR_MAX_ITER = 1000
OLD_AGE_THRESHOLD = 19


## 1. Imports and device setup

In [ ]:
import pickle
import re
import subprocess

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.sparse as sp
import torch

from IPython.display import display
from scipy.io import mmread
from sklearn.decomposition import PCA, NMF
from sklearn.linear_model import LinearRegression
from torch_scatter import scatter_mean

from SpiderNet.config import *
from SpiderNet.utils import scatter_nanmean
from SpiderNet.io import load_processed_data
from SpiderNet.api import build_model, infer_meta_interactions, normalize_outputs, export_results

cuda_available = torch.cuda.is_available()
device = "cuda" if cuda_available else "cpu"
print(f"Using device: {device}")

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["mathtext.fontset"] = "dejavuserif"
plt.rcParams["font.family"] = "arial"
sns.set_theme(style="white", context="paper")

## 2. Build path configs

In [ ]:

train_paths = PathConfig(
    data_root=TRAIN_DATA_ROOT,
    output_root=TRAIN_OUTPUT_ROOT,
    species=SPECIES,
    version=VERSION,
)

sagittal_paths = PathConfig(
    data_root=SAGITTAL_DATA_ROOT,
    output_root=SAGITTAL_OUTPUT_ROOT,
    species=SPECIES,
    version=VERSION,
)

hippocampus_paths = PathConfig(
    data_root=HIPPOCAMPUS_DATA_ROOT,
    output_root=HIPPOCAMPUS_OUTPUT_ROOT,
    species=SPECIES,
    version=VERSION,
)

preprocess_cfg = PreprocessConfig(
    n_hvg=N_HVG,
    n_hvg_lr=N_HVG_LR,
    num_neighbors=NUM_NEIGHBORS
)

train_cfg = TrainingConfig(
    dim_envir=DIM_ENVIR,
    n_jobs=N_JOBS,
    max_epoch=MAX_EPOCH,
    version=VERSION
)

train_run_dirs = train_paths.ensure_dirs(dim_envir=train_cfg.dim_envir)
sagittal_run_dirs = sagittal_paths.ensure_dirs(dim_envir=train_cfg.dim_envir)
hippocampus_run_dirs = hippocampus_paths.ensure_dirs(dim_envir=train_cfg.dim_envir)

display(pd.DataFrame([
    {"dataset": "train", "data_root": str(train_paths.data_root), "output_root": str(train_paths.output_root), "run_dir": str(train_run_dirs["run_dir"])},
    {"dataset": "sagittal", "data_root": str(sagittal_paths.data_root), "output_root": str(sagittal_paths.output_root), "run_dir": str(sagittal_run_dirs["run_dir"])},
    {"dataset": "hippocampus", "data_root": str(hippocampus_paths.data_root), "output_root": str(hippocampus_paths.output_root), "run_dir": str(hippocampus_run_dirs["run_dir"])},
]))


### Quick path check

In [ ]:
rows = [
    {"item": "TRAIN_DATA_ROOT", "value": str(TRAIN_DATA_ROOT), "exists": input_path(TRAIN_DATA_ROOT).exists()},
    {"item": "TRAIN_OUTPUT_ROOT", "value": str(TRAIN_OUTPUT_ROOT), "exists": input_path(TRAIN_OUTPUT_ROOT).exists()},
    {"item": "TRAIN_PROCESSED_DATA_DIR", "value": str(TRAIN_PROCESSED_DATA_DIR), "exists": input_path(TRAIN_PROCESSED_DATA_DIR).exists()},
    {"item": "SAGITTAL_DATA_ROOT", "value": str(SAGITTAL_DATA_ROOT), "exists": input_path(SAGITTAL_DATA_ROOT).exists()},
    {"item": "SAGITTAL_PROCESSED_DATA_DIR", "value": str(SAGITTAL_PROCESSED_DATA_DIR), "exists": input_path(SAGITTAL_PROCESSED_DATA_DIR).exists()},
    {"item": "HIPPOCAMPUS_DATA_ROOT", "value": str(HIPPOCAMPUS_DATA_ROOT), "exists": input_path(HIPPOCAMPUS_DATA_ROOT).exists()},
    {"item": "HIPPOCAMPUS_PROCESSED_DATA_DIR", "value": str(HIPPOCAMPUS_PROCESSED_DATA_DIR), "exists": input_path(HIPPOCAMPUS_PROCESSED_DATA_DIR).exists()},
    {"item": "Train CellChat DB", "value": str(train_paths.cellchat_db), "exists": input_path(Path(train_paths.cellchat_db)).exists()},
    {"item": "Train scSeqComm DB", "value": str(train_paths.scseqcomm_db), "exists": input_path(Path(train_paths.scseqcomm_db)).exists()},
]
display(pd.DataFrame(rows))


## 3. Load the training processed data and the trained SpiderNet checkpoint

In [ ]:
processed_train = load_processed_data(TRAIN_PROCESSED_DATA_DIR)

print("Training processed directory:", TRAIN_PROCESSED_DATA_DIR)
print("Training batches:", len(processed_train.adata_list))
print("Training cells:", np.sum([adata.n_obs for adata in processed_train.adata_list]))
print("Training genes:", processed_train.genenames_train.shape[0])
print("Training LR pairs:", len(processed_train.lr_list))


In [ ]:

def resolve_latest_checkpoint(model_dir, manual_checkpoint=None):
    if manual_checkpoint is not None:
        checkpoint_path = Path(manual_checkpoint)
        if not input_path(checkpoint_path).exists():
            raise FileNotFoundError(f"MODEL_CHECKPOINT does not exist: {checkpoint_path}")
        return checkpoint_path

    checkpoint_candidates = list(Path(model_dir).glob("model_epoch*.pth"))
    if len(checkpoint_candidates) == 0:
        raise FileNotFoundError(f"No checkpoint matching 'model_epoch*.pth' was found in: {model_dir}")

    def extract_epoch(path_obj):
        match = re.search(r"model_epoch(\d+)\.pth$", path_obj.name)
        return int(match.group(1)) if match else -1

    checkpoint_candidates = sorted(checkpoint_candidates, key=extract_epoch)
    return checkpoint_candidates[-1]


checkpoint_path = resolve_latest_checkpoint(train_run_dirs["model_dir"], MODEL_CHECKPOINT)
print("Checkpoint selected:", checkpoint_path)


In [ ]:

model = build_model(
    processed=processed_train,
    train_cfg=train_cfg,
    device=device,
    hidden_channels=HIDDEN_CHANNELS,
)

state = torch.load(checkpoint_path, map_location=device)
if isinstance(state, dict) and "model_state_dict" in state:
    state = state["model_state_dict"]

model.load_state_dict(state)
model = model.to(device)
model.eval()

print("Model loaded and set to eval mode.")


## 4. Helper functions

In [ ]:
def load_pickle(path):
    with open(input_path(path), "rb") as handle:
        return pickle.load(handle)


def save_pickle(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path(path), "wb") as handle:
        pickle.dump(obj, handle)


def get_dense_gene_matrix(adata_list):
    matrices = []
    for adata in adata_list:
        X = adata.X.toarray() if hasattr(adata.X, "toarray") else np.asarray(adata.X)
        matrices.append(X)
    return np.vstack(matrices)


def get_obs_concat(processed, candidates, required=False):
    if isinstance(candidates, str):
        candidates = [candidates]

    for column in candidates:
        values = []
        ok = True
        for adata in processed.adata_list:
            if column not in adata.obs.columns:
                ok = False
                break
            values.append(np.asarray(adata.obs[column]))
        if ok:
            return np.hstack(values), column

    if required:
        raise KeyError(f"None of the candidate columns were found: {candidates}")
    return None, None


def get_single_obs_values(adata, candidates):
    values = []
    for column in candidates:
        if column not in adata.obs.columns:
            continue
        unique_values = pd.unique(adata.obs[column].astype(str))
        for value in unique_values:
            value = str(value).strip()
            if len(value) > 0 and value.lower() != "nan":
                values.append(value)
    seen = []
    for value in values:
        if value not in seen:
            seen.append(value)
    return seen


def format_numeric_token(value):
    try:
        value_float = float(value)
        if np.isfinite(value_float) and abs(value_float - round(value_float)) < 1e-8:
            return str(int(round(value_float)))
        return str(value_float)
    except Exception:
        return str(value)


def get_slice_identifier_candidates(adata):
    raw_values = get_single_obs_values(
        adata,
        ["age", "sample_name_full", "sample_name", "sample", "SAMPLE_ID"]
    )

    candidates = []
    for value in raw_values:
        value_str = str(value)
        candidates.append(value_str)
        candidates.append(format_numeric_token(value_str))
        candidates.append(value_str.replace(" ", "_"))
        candidates.append(value_str.replace("-", "_"))

    obs_name_hint = None
    if getattr(adata, "obs_names", None) is not None and len(adata.obs_names) > 0:
        obs_name_hint = str(adata.obs_names[0]).split("_")[0]
    if obs_name_hint:
        candidates.append(obs_name_hint)

    cleaned = []
    for value in candidates:
        value = str(value).strip()
        if len(value) == 0 or value.lower() == "nan":
            continue
        if value not in cleaned:
            cleaned.append(value)
    return cleaned


def aggregate_mi_features(factor_envir_list, spidernet_data_list, device):
    mi_sr_agg_all = []
    for factor_envir_cur, data_cur in zip(factor_envir_list, spidernet_data_list):
        factor_envir_cur = torch.tensor(factor_envir_cur, dtype=torch.float32, device=device)
        edge_index_cur = data_cur["edge_index"].to(device)
        num_cell_cur = data_cur.x.shape[0]

        mi_receiver_agg_cur = scatter_mean(
            factor_envir_cur,
            edge_index_cur[:, 1].to(torch.int64),
            dim=0,
            dim_size=num_cell_cur
        ).cpu().numpy()

        mi_sender_agg_cur = scatter_mean(
            factor_envir_cur,
            edge_index_cur[:, 0].to(torch.int64),
            dim=0,
            dim_size=num_cell_cur
        ).cpu().numpy()

        mi_sr_agg_cur = np.hstack([mi_sender_agg_cur, mi_receiver_agg_cur])
        mi_sr_agg_all.append(mi_sr_agg_cur)

    return np.vstack(mi_sr_agg_all)


def fit_celltype_regressors(X, celltype_all, y):
    regressors = {}
    unique_celltypes = np.sort(pd.unique(celltype_all))
    for celltype_cur in unique_celltypes:
        idx = np.where(celltype_all == celltype_cur)[0]
        if idx.shape[0] == 0:
            continue
        regressors[celltype_cur] = LinearRegression().fit(X[idx, :], y[idx])
    return regressors


def get_commot_pathway_keys(commot_adata):
    keys = list(commot_adata.obsp.keys())
    total_key = "commot-cellchat-total-total"
    pathway_keys = [
        k for k in keys
        if k.startswith("commot-cellchat-")
        and len(k.split("-")) == 3
        and k != total_key
    ]
    return sorted(pathway_keys)


def get_banksy_lambda_string(value):
    value_str = str(value)
    if "." in value_str:
        value_str = value_str.rstrip("0").rstrip(".")
    return value_str


def resolve_banksy_triplet_for_slice(adata, dataset_label, dataset_cfg):
    banksy_dir = Path(dataset_cfg["output_root"]) / BANKSY_SUBDIR
    if not input_path(banksy_dir).exists():
        raise FileNotFoundError(f"{dataset_label}: Banksy directory does not exist: {banksy_dir}")

    lambda_str = get_banksy_lambda_string(BANKSY_LAMBDA_CELLIDENTITY)
    filename_hint = dataset_cfg["filename_hint"]
    identifier_candidates = get_slice_identifier_candidates(adata)

    preferred_patterns = []
    for token in identifier_candidates:
        preferred_patterns.extend([
            f"aging_{filename_hint}_age{token}_Banksy_cellidentity_lambda{lambda_str}.mtx",
            f"aging_{filename_hint}_{token}_Banksy_cellidentity_lambda{lambda_str}.mtx",
            f"*{filename_hint}*{token}*Banksy_cellidentity_lambda{lambda_str}.mtx",
            f"*{token}*Banksy_cellidentity_lambda{lambda_str}.mtx",
        ])

    candidate_paths = []
    for pattern in preferred_patterns:
        candidate_paths.extend(sorted(banksy_dir.glob(pattern)))

    if len(candidate_paths) == 0:
        candidate_paths = sorted(banksy_dir.glob(f"*Banksy_cellidentity_lambda{lambda_str}.mtx"))

    candidate_paths_unique = []
    for path in candidate_paths:
        if path not in candidate_paths_unique:
            candidate_paths_unique.append(path)

    if len(candidate_paths_unique) == 0:
        raise FileNotFoundError(
            f"{dataset_label}: no Banksy matrix file was found in {banksy_dir} for identifiers {identifier_candidates}"
        )

    obs_index = pd.Index(adata.obs.index.astype(str))
    best_score = None
    best_triplet = None

    for mtx_path in candidate_paths_unique:
        genes_path = Path(str(mtx_path).replace(".mtx", "_genes.csv"))
        barcodes_path = Path(str(mtx_path).replace(".mtx", "_barcodes.csv"))
        if not input_path(genes_path).exists() or not input_path(barcodes_path).exists():
            continue

        barcodes_df = pd.read_csv(input_path(barcodes_path))
        barcode_col = "barcode" if "barcode" in barcodes_df.columns else barcodes_df.columns[0]
        barcodes = pd.Index(barcodes_df[barcode_col].astype(str))
        overlap = int(obs_index.isin(barcodes).sum())
        contains_hint = int(filename_hint.lower() in mtx_path.name.lower())
        token_hit = int(any(token.lower() in mtx_path.name.lower() for token in identifier_candidates))
        n_obs_match = int(len(barcodes) == adata.n_obs)
        score = (overlap, n_obs_match, token_hit, contains_hint)

        if (best_score is None) or (score > best_score):
            best_score = score
            best_triplet = (mtx_path, genes_path, barcodes_path)

    if best_triplet is None or best_score[0] == 0:
        raise FileNotFoundError(
            f"{dataset_label}: could not match a Banksy file to slice identifiers {identifier_candidates} in {banksy_dir}"
        )

    return best_triplet


def load_banksy_matrix_for_processed(processed_transfer, dataset_label, dataset_cfg):
    banksy_matrix_all = []

    for slice_index, adata_cur in enumerate(processed_transfer.adata_list):
        mtx_path, genes_path, barcodes_path = resolve_banksy_triplet_for_slice(
            adata=adata_cur,
            dataset_label=dataset_label,
            dataset_cfg=dataset_cfg,
        )

        print(f"{dataset_label}: matched Banksy slice {slice_index} -> {mtx_path.name}")

        mat = mmread(input_path(mtx_path)).tocsr()
        genes_df = pd.read_csv(input_path(genes_path))
        barcodes_df = pd.read_csv(input_path(barcodes_path))

        gene_col = "gene" if "gene" in genes_df.columns else genes_df.columns[0]
        barcode_col = "barcode" if "barcode" in barcodes_df.columns else barcodes_df.columns[0]

        genes = genes_df[gene_col].astype(str).tolist()
        barcodes = barcodes_df[barcode_col].astype(str).tolist()

        banksy_matrix = pd.DataFrame.sparse.from_spmatrix(
            mat,
            index=genes,
            columns=barcodes,
        ).T
        banksy_matrix.index = banksy_matrix.index.astype(str)

        obs_index = pd.Index(adata_cur.obs.index.astype(str))
        missing_barcodes = obs_index.difference(banksy_matrix.index)
        if len(missing_barcodes) > 0:
            raise KeyError(
                f"{dataset_label}: {len(missing_barcodes)} cells are missing from the matched Banksy file "
                f"{mtx_path.name}. First few missing barcodes: {missing_barcodes[:5].tolist()}"
            )

        banksy_matrix_all.append(banksy_matrix.loc[obs_index, :])

    banksy_matrix_all = pd.concat(banksy_matrix_all, axis=0)
    banksy_matrix_all = banksy_matrix_all.sparse.to_dense() if hasattr(banksy_matrix_all, "sparse") else banksy_matrix_all
    return np.asarray(banksy_matrix_all, dtype=np.float32)




def resolve_commot_file_for_slice(adata, dataset_label, dataset_cfg, allow_missing=False):
    commot_dir = Path(dataset_cfg["output_root"]) / COMMOT_SUBDIR
    if not input_path(commot_dir).exists():
        if allow_missing:
            return None
        raise FileNotFoundError(f"{dataset_label}: COMMOT directory does not exist: {commot_dir}")

    filename_hint = dataset_cfg["filename_hint"]
    identifier_candidates = get_slice_identifier_candidates(adata)

    preferred_patterns = []
    for token in identifier_candidates:
        preferred_patterns.extend([
            f"aging_{filename_hint}_age{token}_commot.h5ad",
            f"aging_{filename_hint}_{token}_commot.h5ad",
            f"*{filename_hint}*{token}*commot*.h5ad",
            f"*{token}*commot*.h5ad",
        ])

    candidate_paths = []
    for pattern in preferred_patterns:
        candidate_paths.extend(sorted(commot_dir.glob(pattern)))

    if len(candidate_paths) == 0:
        candidate_paths = sorted(commot_dir.glob("*commot*.h5ad"))

    candidate_paths_unique = []
    for path in candidate_paths:
        if path not in candidate_paths_unique:
            candidate_paths_unique.append(path)

    if len(candidate_paths_unique) == 0:
        if allow_missing:
            return None
        raise FileNotFoundError(
            f"{dataset_label}: no COMMOT file was found in {commot_dir} for identifiers {identifier_candidates}"
        )

    best_score = None
    best_path = None

    for commot_path in candidate_paths_unique:
        try:
            adata_commot = sc.read_h5ad(input_path(commot_path), backed="r")
            n_obs_commot = int(adata_commot.n_obs)
            obs_names_commot = pd.Index(adata_commot.obs_names.astype(str))
            try:
                adata_commot.file.close()
            except Exception:
                pass
        except Exception:
            adata_commot = sc.read_h5ad(input_path(commot_path))
            n_obs_commot = int(adata_commot.n_obs)
            obs_names_commot = pd.Index(adata_commot.obs_names.astype(str))

        obs_index = pd.Index(adata.obs.index.astype(str))
        overlap = int(obs_index.isin(obs_names_commot).sum())
        n_obs_match = int(n_obs_commot == adata.n_obs)
        contains_hint = int(filename_hint.lower() in commot_path.name.lower())
        token_hit = int(any(token.lower() in commot_path.name.lower() for token in identifier_candidates))
        score = (overlap, n_obs_match, token_hit, contains_hint)

        if (best_score is None) or (score > best_score):
            best_score = score
            best_path = commot_path

    if best_path is None or best_score[0] == 0:
        if allow_missing:
            return None
        raise FileNotFoundError(
            f"{dataset_label}: could not match a COMMOT file to slice identifiers {identifier_candidates} in {commot_dir}"
        )

    return best_path










def required_processed_bundle_files():
    return [
        "adata_all.h5ad",
        "SpiderNet_data_pyg_list.pkl",
        "LR_list.pkl",
        "LR_list_all.pkl",
        "LR_list_cellchatdb.pkl",
        "LR_meta_cellchatdb.pkl",
        "batch_cell_unique.pkl",
        "batch_cell.pkl",
        "genenames.pkl",
        "genenames_train.pkl",
        "adata_list.pkl",
        "cellclass_unique.pkl",
    ]


def sync_processed_bundle(source_dir, target_dir, overwrite=False):
    import shutil
    source_dir = Path(source_dir)
    target_dir = Path(target_dir)
    if not input_path(source_dir).exists():
        raise FileNotFoundError(f"Processed-data directory does not exist: {source_dir}")

    missing_required = [name for name in required_processed_bundle_files() if not (input_path(source_dir / name)).exists()]
    if missing_required:
        raise FileNotFoundError(
            f"Processed-data directory is missing required files: {missing_required}. Source: {source_dir}"
        )

    target_dir.mkdir(parents=True, exist_ok=True)
    copied = []
    for item in source_dir.iterdir():
        if not item.is_file():
            continue
        dest = target_dir / item.name
        if overwrite or not input_path(dest).exists():
            shutil.copy2(item, dest)
            copied.append(item.name)

    if len(copied) > 0:
        print(f"Copied {len(copied)} files from {source_dir} to {target_dir}")
    else:
        print(f"Processed bundle already available at: {target_dir}")


def preprocess_sagittal_dataset(processed_dir, run_dirs_obj):
    processed_file = Path(run_dirs_obj["run_dir"]) / "SpiderNet_data_pyg_list.pkl"
    if input_path(processed_file).exists() and not REPROCESS_TRANSFER_DATA:
        print(f"Using existing processed sagittal data: {processed_file}")
        return

    sync_processed_bundle(
        source_dir=processed_dir,
        target_dir=run_dirs_obj["run_dir"],
        overwrite=REPROCESS_TRANSFER_DATA,
    )


def preprocess_hippocampus_dataset(processed_dir, run_dirs_obj):
    processed_file = Path(run_dirs_obj["run_dir"]) / "SpiderNet_data_pyg_list.pkl"
    if input_path(processed_file).exists() and not REPROCESS_TRANSFER_DATA:
        print(f"Using existing processed hippocampus data: {processed_file}")
        return

    sync_processed_bundle(
        source_dir=processed_dir,
        target_dir=run_dirs_obj["run_dir"],
        overwrite=REPROCESS_TRANSFER_DATA,
    )


def align_transfer_cellclass_space(processed_transfer, processed_train):
    reference_cell_classes = np.asarray(processed_train.spidernet_data[0]["cell_class_unique"])
    reference_index = {label: idx for idx, label in enumerate(reference_cell_classes)}

    for batch_idx, data_cur in enumerate(processed_transfer.spidernet_data):
        current_unique = np.asarray(data_cur["cell_class_unique"])
        current_onehot = data_cur["cell_class_onehot"].cpu().numpy()
        current_labels = current_unique[np.argmax(current_onehot, axis=1)]

        new_onehot = np.zeros((len(current_labels), len(reference_cell_classes)), dtype=np.float32)
        missing_labels = set()

        for row_idx, label in enumerate(current_labels):
            if label in reference_index:
                new_onehot[row_idx, reference_index[label]] = 1.0
            else:
                missing_labels.add(label)

        data_cur["cell_class_onehot"] = torch.tensor(new_onehot, dtype=torch.float32)
        data_cur["cell_class_unique"] = reference_cell_classes

        if len(missing_labels) > 0:
            print(
                f"Batch {batch_idx}: these transfer cell types are not present in the training cell-class space "
                f"and were left as all-zero one-hot rows: {sorted(missing_labels)}"
            )

    return processed_transfer


def infer_transfer_results(model, processed_transfer, processed_train, output_dir, reference_processed_data_dir):
    processed_transfer = align_transfer_cellclass_space(processed_transfer, processed_train)
    transfer_results = infer_meta_interactions(model=model, processed=processed_transfer)
    transfer_results = normalize_outputs(transfer_results)
    export_results(
        results=transfer_results,
        processed=processed_transfer,
        precessed_data_dir=reference_processed_data_dir,
        output_dir=output_path(output_dir),
    )
    return processed_transfer, transfer_results


def predict_age_from_features(
    feature_matrix,
    processed_transfer,
    regressors,
    method_name,
    dataset_label,
    output_dir,
    old_age_threshold=19,
    hippocampus_old_keyword="Hippocampus_O",
):
    celltype_all, celltype_col = get_obs_concat(processed_transfer, ["celltype", "CELL_TYPE"], required=True)
    age_all, age_col = get_obs_concat(processed_transfer, ["age", "SAMPLE_ID"], required=False)
    sample_all, sample_col = get_obs_concat(
        processed_transfer,
        ["sample_name_full", "sample_name", "sample", "SAMPLE_ID"],
        required=False
    )

    pred_rows = []
    skipped_celltypes = []

    for celltype_cur in np.sort(pd.unique(celltype_all)):
        idx = np.where(celltype_all == celltype_cur)[0]
        if celltype_cur not in regressors:
            skipped_celltypes.append(celltype_cur)
            continue

        pred_df_cur = pd.DataFrame({
            "cell_index": idx.astype(int),
            "age_predicted": regressors[celltype_cur].predict(feature_matrix[idx, :]),
            "celltype": celltype_all[idx],
        })

        if age_all is not None:
            pred_df_cur["true_age"] = age_all[idx]
        if sample_all is not None:
            pred_df_cur["sample"] = sample_all[idx]

        pred_rows.append(pred_df_cur)

    if len(skipped_celltypes) > 0:
        print(f"{dataset_label} / {method_name}: skipped cell types without a matching trained regressor:")
        print(sorted(skipped_celltypes))

    if len(pred_rows) == 0:
        raise ValueError(f"No predictions were generated for {dataset_label} / {method_name}.")

    pred_df = pd.concat(pred_rows, axis=0).sort_values("cell_index").reset_index(drop=True)

    if "true_age" in pred_df.columns:
        true_age_numeric = pd.to_numeric(pred_df["true_age"], errors="coerce")
        if true_age_numeric.notna().all():
            pred_df["true_age"] = true_age_numeric
            pred_df["AgeGroup"] = np.where(pred_df["true_age"] > old_age_threshold, "Old", "Young")
        elif "sample" in pred_df.columns:
            pred_df["AgeGroup"] = np.where(
                pred_df["sample"].astype(str).str.contains(hippocampus_old_keyword),
                "Old",
                "Young"
            )
        else:
            raise ValueError(f"Could not determine AgeGroup for {dataset_label} / {method_name}.")
    elif "sample" in pred_df.columns:
        pred_df["AgeGroup"] = np.where(
            pred_df["sample"].astype(str).str.contains(hippocampus_old_keyword),
            "Old",
            "Young"
        )
    else:
        raise ValueError(f"Could not determine AgeGroup for {dataset_label} / {method_name}.")

    output_dir = Path(output_dir)
    pred_path = output_dir / f"{dataset_label}_predicted_age_{method_name}.csv"
    pred_df.to_csv(output_path(pred_path), index=False)
    print(f"Saved predictions to: {pred_path}")

    return pred_df


def plot_density_by_celltype(pred_df, dataset_label, method_name, output_dir, col_wrap=5, height=1.5):
    df = pred_df.copy()

    g = sns.FacetGrid(
        df,
        col="celltype",
        col_wrap=col_wrap,
        sharex=False,
        sharey=False,
        height=height,
    )
    g.map_dataframe(
        sns.kdeplot,
        x="age_predicted",
        hue="AgeGroup",
        fill=False,
        common_norm=False,
        alpha=0.5,
        palette={"Young": "steelblue", "Old": "orange"},
        warn_singular=False,
    )

    g.set_titles(col_template="{col_name}")
    g.set_axis_labels("Predicted age", "Density")
    plt.suptitle(f"{dataset_label}: predicted age by cell type ({method_name})", y=1.02)
    plt.tight_layout()

    out_path = Path(output_dir) / f"{dataset_label}_AgePrediction_DensityPlot_byCellType_{method_name}.pdf"
    plt.savefig(output_path(out_path), dpi=300, bbox_inches="tight", transparent=True)
    plt.show()
    plt.close()
    print(f"Saved density plot to: {out_path}")


def compute_old_young_ratio(pred_df, output_dir, dataset_label, method_name):
    mean_age_df = (
        pred_df.groupby(["celltype", "AgeGroup"])["age_predicted"]
        .mean()
        .unstack()
    )

    if "Old" not in mean_age_df.columns or "Young" not in mean_age_df.columns:
        raise ValueError(
            f"{dataset_label} / {method_name}: both Old and Young groups are required to compute the ratio."
        )

    mean_age_df["Old_vs_Young_ratio"] = mean_age_df["Old"] / mean_age_df["Young"]
    ratio_df = mean_age_df[["Old_vs_Young_ratio"]].sort_values("Old_vs_Young_ratio", ascending=False)

    out_path = Path(output_dir) / f"{dataset_label}_Old_vs_Young_ratio_{method_name}.csv"
    ratio_df.to_csv(output_path(out_path))
    print(f"Saved ratio table to: {out_path}")

    return ratio_df


def combine_ratio_tables(ratio_df_dict, method_order=None):
    if method_order is None:
        method_order = PREFERRED_METHOD_ORDER

    available_methods = [method for method in method_order if method in ratio_df_dict]
    if len(available_methods) == 0:
        raise ValueError("No ratio tables were provided.")

    common_celltypes = None
    for method in available_methods:
        celltypes_cur = set(ratio_df_dict[method].index.tolist())
        if common_celltypes is None:
            common_celltypes = celltypes_cur
        else:
            common_celltypes = common_celltypes.intersection(celltypes_cur)

    common_celltypes = sorted(common_celltypes)
    ratio_df_combined = pd.DataFrame({"CellType": common_celltypes})

    for method in available_methods:
        ratio_df_combined[f"{method}_Old_vs_Young_ratio"] = (
            ratio_df_dict[method].loc[common_celltypes, "Old_vs_Young_ratio"].values
        )

    return ratio_df_combined


import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from pathlib import Path
from matplotlib.patches import Patch

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["mathtext.fontset"] = "dejavuserif"
plt.rcParams["font.family"] = "arial"


# ==============================================================
# Method colors: consistent with AgingBrain_training_analysis
# --------------------------------------------------------------
# Use different fill and edge colors.
# Banksy is added as a new method.
# PCA is kept as a gray fallback if this old code path still plots PCA.
# ==============================================================
METHOD_FILL_EDGE_COLOR_DICT = {
    "SpiderNet": {"edge": "#9F3B38", "fill": "#E1B6A7"},
    "NMF-LR": {"edge": "#82CCE2", "fill": "#D4ECF1"},
    "COMMOT": {"edge": "#519384", "fill": "#B9CEC7"},
    "ScCChain": {"edge": "#636491", "fill": "#A6A2B9"},
    "Banksy": {"edge": "#8A5A44", "fill": "#D9C1B0"},
    "PCA": {"edge": "#7F7F7F", "fill": "#D0D0D0"},
}

DEFAULT_METHOD_COLOR = {"edge": "#4D4D4D", "fill": "#BDBDBD"}


def plot_ratio_comparison(ratio_df_combined, dataset_label, output_dir, y_min=None, y_max=None):
    ratio_cols = [
        c for c in ratio_df_combined.columns
        if c.endswith("_Old_vs_Young_ratio")
    ]

    method_order = [
        c.replace("_Old_vs_Young_ratio", "")
        for c in ratio_cols
        if c.replace("_Old_vs_Young_ratio", "") in PREFERRED_METHOD_ORDER
    ]
    method_order = [m for m in PREFERRED_METHOD_ORDER if m in method_order]

    if len(method_order) == 0:
        raise ValueError("No ratio columns were found for plotting.")

    sort_col = (
        "SpiderNet_Old_vs_Young_ratio"
        if "SpiderNet_Old_vs_Young_ratio" in ratio_df_combined.columns
        else ratio_cols[0]
    )
    df = ratio_df_combined.copy().sort_values(sort_col, ascending=False)

    celltypes = df["CellType"].values
    n = len(celltypes)
    n_methods = len(method_order)

    group_spacing = 1.8
    total_width = 0.9
    bar_width = total_width / max(n_methods, 1)
    x = np.arange(n) * group_spacing
    offsets = np.linspace(
        -total_width / 2 + bar_width / 2,
        total_width / 2 - bar_width / 2,
        n_methods,
    )

    fig_width = max(7.5, 1.2 * n)
    fig, ax = plt.subplots(figsize=(fig_width, 4), facecolor="white")
    ax.set_facecolor("white")

    for offset, method in zip(offsets, method_order):
        color_cfg = METHOD_FILL_EDGE_COLOR_DICT.get(method, DEFAULT_METHOD_COLOR)

        ax.bar(
            x + offset,
            df[f"{method}_Old_vs_Young_ratio"].values,
            width=bar_width,
            facecolor=color_cfg["fill"],
            edgecolor=color_cfg["edge"],
            linewidth=1.0,
            label=method,
        )

    ax.axhline(1, linestyle="--", color="gray", linewidth=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels(celltypes, rotation=60, ha="right")
    ax.set_ylabel("Old / young mean predicted age")

    if y_min is None:
        ax.set_ylim(0, y_max)
    else:
        ax.set_ylim(y_min, y_max)

    sns.despine(ax=ax, top=True, right=True)

    ax.spines["left"].set_visible(True)
    ax.spines["bottom"].set_visible(True)
    ax.spines["left"].set_color("black")
    ax.spines["bottom"].set_color("black")
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.tick_params(width=0.8, length=3.5)

    legend_handles = [
        Patch(
            facecolor=METHOD_FILL_EDGE_COLOR_DICT.get(m, DEFAULT_METHOD_COLOR)["fill"],
            edgecolor=METHOD_FILL_EDGE_COLOR_DICT.get(m, DEFAULT_METHOD_COLOR)["edge"],
            linewidth=1.0,
            label=m,
        )
        for m in method_order
    ]

    ax.legend(
        handles=legend_handles,
        frameon=False,
    )

    plt.tight_layout()

    method_suffix = "_vs_".join(method_order)
    out_path = Path(output_dir) / f"{dataset_label}_Old_vs_Young_ratio_barplot_{method_suffix}.pdf"
    plt.savefig(output_path(out_path), bbox_inches="tight", transparent=True)
    plt.show()
    plt.close()

    print(f"Saved ratio bar plot to: {out_path}")




In [ ]:

# =========================================================
# Coronal-reference baseline definitions: NMF-LR, COMMOT, and Banksy
# ---------------------------------------------------------
# These definitions supersede the generic helper implementations above.
#
# Transfer-safe rules:
#   * SpiderNet: use the Coronal-trained SpiderNet model to infer fixed MI dimensions.
#   * NMF-LR: fit the NMF-LR basis on Coronal edge-level LR features, then only transform external slices.
#   * COMMOT: use Coronal COMMOT feature order; external missing pathways are filled with 0 and extras are dropped.
#   * Banksy: fit the Banksy PCA basis on Coronal Banksy matrix, then only transform aligned external Banksy matrices.
#
# The gene-expression PCA baseline is intentionally not run or displayed.
# =========================================================

import scipy.sparse as sp
from sklearn.decomposition import NMF, PCA


# -----------------------------
# NMF-LR: fit on Coronal, transform external
# -----------------------------
def _nmflr_get_edge_index_tensor(edge_index, device):
    if torch.is_tensor(edge_index):
        edge_index_t = edge_index.detach().clone().to(torch.int64)
    else:
        edge_index_t = torch.as_tensor(edge_index, dtype=torch.int64)

    if edge_index_t.ndim != 2:
        raise ValueError(f"edge_index must be 2-dimensional, got shape {tuple(edge_index_t.shape)}")

    if edge_index_t.shape[0] == 2 and edge_index_t.shape[1] != 2:
        edge_index_t = edge_index_t.T

    if edge_index_t.shape[1] != 2:
        raise ValueError(f"edge_index must have two columns after conversion, got shape {tuple(edge_index_t.shape)}")

    return edge_index_t.to(device)


def _clean_nonnegative_matrix_for_nmf(X):
    """Convert edge-level LR coexpression matrix to a non-negative matrix for sklearn NMF."""
    if torch.is_tensor(X):
        X = X.detach().cpu().numpy()

    if sp.issparse(X):
        X = X.tocsr(copy=True)
        X.data = np.nan_to_num(X.data, nan=0.0, posinf=0.0, neginf=0.0)
        X.data[X.data < 0] = 0.0
        return X

    X = np.asarray(X, dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    X[X < 0] = 0.0
    return X


def _nmflr_to_feature_name_list(value, n_features):
    """Best-effort conversion of stored LR feature names to a clean string list."""
    if value is None:
        return None
    if torch.is_tensor(value):
        value = value.detach().cpu().numpy()
    if hasattr(value, "tolist") and not isinstance(value, (list, tuple, pd.Index)):
        value = value.tolist()
    if isinstance(value, pd.Index):
        value = value.tolist()
    if isinstance(value, (list, tuple, np.ndarray)):
        names = [str(x) for x in list(value)]
        if len(names) == n_features:
            return names
    return None


def _nmflr_get_lr_feature_names_from_data(data_cur, n_features):
    """
    Try to recover the LR-pair / feature names associated with
    data_cur['cellpair_LRpair_neigh'] columns.

    If names are unavailable, fall back to positional names. The positional
    fallback can still pad/truncate external matrices to the Coronal NMF input
    dimension, but true name-based alignment is only possible when the processed
    object stores LR feature names.
    """
    candidate_keys = [
        "cellpair_LRpair_neigh_names",
        "cellpair_LRpair_neigh_columns",
        "cellpair_LRpair_names",
        "cellpair_lrpair_names",
        "LRpair_names",
        "LR_pair_names",
        "lrpair_names",
        "lr_pair_names",
        "ligand_receptor_pair_names",
        "ligand_receptor_names",
        "LRpair_list",
        "lrpair_list",
        "feature_names",
    ]

    # PyG Data supports both dict-like and attribute-like access.
    for key in candidate_keys:
        try:
            if key in data_cur:
                names = _nmflr_to_feature_name_list(data_cur[key], n_features)
                if names is not None:
                    return names
        except Exception:
            pass

        try:
            if hasattr(data_cur, key):
                names = _nmflr_to_feature_name_list(getattr(data_cur, key), n_features)
                if names is not None:
                    return names
        except Exception:
            pass

    # Common pattern: ligand and receptor columns stored separately.
    ligand_keys = ["ligands", "ligand_names", "Ligand", "Ligand_gene", "ligand_gene"]
    receptor_keys = ["receptors", "receptor_names", "Receptor", "Receptor_gene", "receptor_gene"]
    for lk in ligand_keys:
        for rk in receptor_keys:
            lig = rec = None
            try:
                if lk in data_cur:
                    lig = _nmflr_to_feature_name_list(data_cur[lk], n_features)
            except Exception:
                pass
            try:
                if rk in data_cur:
                    rec = _nmflr_to_feature_name_list(data_cur[rk], n_features)
            except Exception:
                pass
            if lig is not None and rec is not None:
                return [f"{l}__{r}" for l, r in zip(lig, rec)]

    return [f"LR_feature_{i}" for i in range(n_features)]





def split_edge_factor_matrix(factor_all, edge_counts):
    factor_list = []
    start = 0
    for slice_index, n_edges in enumerate(edge_counts):
        end = start + n_edges
        factor_cur = factor_all[start:end].copy()
        if factor_cur.shape[0] != n_edges:
            raise ValueError(
                f"Unexpected split size for slice {slice_index}: {factor_cur.shape[0]} vs expected {n_edges}."
            )
        factor_list.append(factor_cur)
        print(f"Split factors for slice {slice_index + 1}: {factor_cur.shape}")
        start = end

    if start != factor_all.shape[0]:
        raise ValueError(f"Split mismatch: used {start} rows, but factor_all has {factor_all.shape[0]} rows.")

    return factor_list


def fit_training_nmflr_model(spidernet_data_list, n_components=None, random_state=0, max_iter=1000):
    """Fit one Coronal NMF-LR model and return normalized Coronal edge factors.

    The fitted model stores `lr_feature_names_in_`, which is used to align
    external LR matrices before NMF transform.
    """
    if n_components is None:
        n_components = int(globals().get("NMF_LR_N_COMPONENTS", globals().get("DIM_ENVIR", 30)))

    lr_all, edge_counts, lr_feature_names = collect_lr_coexpression_matrix(
        spidernet_data_list,
        return_feature_names=True,
    )

    print("Fitting Coronal NMF-LR basis on concatenated edge-level LR coexpression matrix:", lr_all.shape)
    print("NMF-LR rank:", n_components)

    nmf_lr_model = NMF(
        n_components=int(n_components),
        init="nndsvda",
        random_state=random_state,
        max_iter=max_iter,
    )
    factor_all = nmf_lr_model.fit_transform(lr_all)

    # Store LR feature names inside the fitted sklearn model so external slices
    # can be aligned before transform.
    nmf_lr_model.lr_feature_names_in_ = list(lr_feature_names)

    # Save Coronal scaling only; external factors are normalized using these same values.
    factor_all_colmax = np.max(factor_all, axis=0)
    factor_all_colmax[(~np.isfinite(factor_all_colmax)) | (factor_all_colmax == 0)] = 1.0
    factor_all = (factor_all / factor_all_colmax).astype(np.float32, copy=False)

    factor_lr_list = split_edge_factor_matrix(factor_all, edge_counts)
    return factor_lr_list, nmf_lr_model, factor_all_colmax.astype(np.float32, copy=False)


def aggregate_edge_program_features(factor_list, spidernet_data_list, device):
    """Aggregate edge-level features to [sender aggregate | receiver aggregate] cell-level features."""
    sr_agg_all = []

    for slice_index, (factor_cur, data_cur) in enumerate(zip(factor_list, spidernet_data_list)):
        edge_index_cur = _nmflr_get_edge_index_tensor(data_cur["edge_index"], device=device)
        num_cell_cur = data_cur.x.shape[0]
        factor_cur_t = torch.as_tensor(factor_cur, dtype=torch.float32, device=device)

        if factor_cur_t.shape[0] != edge_index_cur.shape[0]:
            raise ValueError(
                f"Feature rows and edge rows do not match for slice {slice_index}: "
                f"{factor_cur_t.shape[0]} vs {edge_index_cur.shape[0]}."
            )

        receiver_agg_cur = scatter_nanmean(
            factor_cur_t,
            edge_index_cur[:, 1],
            dim=0,
            dim_size=num_cell_cur,
        ).to("cpu").numpy()

        sender_agg_cur = scatter_nanmean(
            factor_cur_t,
            edge_index_cur[:, 0],
            dim=0,
            dim_size=num_cell_cur,
        ).to("cpu").numpy()

        sr_agg_cur = np.hstack([sender_agg_cur, receiver_agg_cur])
        sr_agg_all.append(sr_agg_cur)
        print(f"Aggregated sender/receiver features for slice {slice_index + 1}: {sr_agg_cur.shape}")

    sr_agg_all = np.vstack(sr_agg_all)
    sr_agg_all = np.nan_to_num(sr_agg_all, nan=0.0, posinf=0.0, neginf=0.0)
    return sr_agg_all


def aggregate_nmflr_features(factor_lr_list, spidernet_data_list, device):
    nmflr_sr_agg_all = aggregate_edge_program_features(
        factor_list=factor_lr_list,
        spidernet_data_list=spidernet_data_list,
        device=device,
    )
    print("NMF_LR_SR_agg_all shape:", nmflr_sr_agg_all.shape)
    return nmflr_sr_agg_all


# -----------------------------
# Banksy: fit PCA on Coronal matrix, transform aligned external matrices
# -----------------------------
def load_banksy_matrix_dataframe_for_processed(processed_transfer, dataset_label, dataset_cfg):
    banksy_df_parts = []

    for slice_index, adata_cur in enumerate(processed_transfer.adata_list):
        mtx_path, genes_path, barcodes_path = resolve_banksy_triplet_for_slice(
            adata=adata_cur,
            dataset_label=dataset_label,
            dataset_cfg=dataset_cfg,
        )
        print(f"{dataset_label}: matched Banksy slice {slice_index} -> {mtx_path.name}")

        mat = mmread(input_path(mtx_path)).tocsr()
        genes_df = pd.read_csv(input_path(genes_path))
        barcodes_df = pd.read_csv(input_path(barcodes_path))

        gene_col = "gene" if "gene" in genes_df.columns else genes_df.columns[0]
        barcode_col = "barcode" if "barcode" in barcodes_df.columns else barcodes_df.columns[0]

        genes = genes_df[gene_col].astype(str).tolist()
        barcodes = barcodes_df[barcode_col].astype(str).tolist()

        banksy_matrix = pd.DataFrame.sparse.from_spmatrix(
            mat,
            index=genes,
            columns=barcodes,
        ).T
        banksy_matrix.index = banksy_matrix.index.astype(str)
        banksy_matrix.columns = banksy_matrix.columns.astype(str)

        obs_index = pd.Index(adata_cur.obs.index.astype(str))
        missing_barcodes = obs_index.difference(banksy_matrix.index)
        if len(missing_barcodes) > 0:
            raise KeyError(
                f"{dataset_label}: {len(missing_barcodes)} cells are missing from the matched Banksy file "
                f"{mtx_path.name}. First few missing barcodes: {missing_barcodes[:5].tolist()}"
            )

        banksy_df_parts.append(banksy_matrix.loc[obs_index, :])

    banksy_df_all = pd.concat(banksy_df_parts, axis=0)
    if hasattr(banksy_df_all, "sparse"):
        banksy_df_all = banksy_df_all.sparse.to_dense()
    banksy_df_all = banksy_df_all.astype(np.float32)
    return banksy_df_all


def fit_training_banksy_pca(processed_train, dataset_cfg, n_components=20):
    banksy_train_df = load_banksy_matrix_dataframe_for_processed(
        processed_transfer=processed_train,
        dataset_label="Coronal",
        dataset_cfg=dataset_cfg,
    )
    banksy_feature_names = banksy_train_df.columns.astype(str).tolist()
    pca_model = PCA(n_components=n_components)
    banksy_features_train = pca_model.fit_transform(banksy_train_df.to_numpy(dtype=np.float32))
    return banksy_features_train, pca_model, banksy_feature_names


def transform_banksy_with_training_pca(processed_transfer, dataset_label, dataset_cfg, banksy_pca_model, banksy_feature_names):
    banksy_df = load_banksy_matrix_dataframe_for_processed(
        processed_transfer=processed_transfer,
        dataset_label=dataset_label,
        dataset_cfg=dataset_cfg,
    )
    # Align to Coronal feature order: missing features are 0, extra features are dropped.
    banksy_df = banksy_df.reindex(columns=pd.Index(banksy_feature_names).astype(str), fill_value=0.0)
    banksy_features = banksy_pca_model.transform(banksy_df.to_numpy(dtype=np.float32))
    return banksy_features


# -----------------------------
# COMMOT: use Coronal feature order for all transfer datasets
# -----------------------------
def _commot_feature_names_from_pathway_keys(pathway_keys):
    return (
        [f"receiver_{k.replace('commot-cellchat-', '')}" for k in pathway_keys]
        + [f"sender_{k.replace('commot-cellchat-', '')}" for k in pathway_keys]
    )


def _commot_pathway_keys_from_feature_names(feature_names):
    receiver_names = []
    for name in feature_names:
        name = str(name)
        if name.startswith("receiver_"):
            receiver_names.append(name.replace("receiver_", "", 1))
    pathway_keys = []
    for name in receiver_names:
        if name.startswith("commot-"):
            pathway_keys.append(name)
        else:
            pathway_keys.append(f"commot-cellchat-{name}")
    return pathway_keys


def load_commot_aggregated_features_transfer_safe(
    processed_transfer,
    dataset_label,
    dataset_cfg,
    allow_missing=False,
    reference_feature_names=None,
):
    if not dataset_cfg.get("has_commot", False):
        if allow_missing:
            return None, None
        raise FileNotFoundError(f"{dataset_label}: COMMOT is marked as unavailable in DATASET_FEATURE_CONFIG.")

    matched_commot_paths = []
    for slice_index, adata_cur in enumerate(processed_transfer.adata_list):
        commot_path = resolve_commot_file_for_slice(
            adata=adata_cur,
            dataset_label=dataset_label,
            dataset_cfg=dataset_cfg,
            allow_missing=allow_missing,
        )
        if commot_path is None:
            if allow_missing:
                return None, None
            raise FileNotFoundError(f"{dataset_label}: failed to resolve COMMOT file for slice {slice_index}")
        print(f"{dataset_label}: matched COMMOT slice {slice_index} -> {commot_path.name}")
        matched_commot_paths.append(commot_path)

    if reference_feature_names is None:
        pathway_union_set = set()
        for commot_path in matched_commot_paths:
            commot_adata = sc.read_h5ad(input_path(commot_path))
            pathway_union_set.update(get_commot_pathway_keys(commot_adata))
        pathway_keys = sorted(pathway_union_set)
        commot_feature_names = _commot_feature_names_from_pathway_keys(pathway_keys)
    else:
        commot_feature_names = [str(x) for x in reference_feature_names]
        pathway_keys = _commot_pathway_keys_from_feature_names(commot_feature_names)

    pathway_to_col = {k: i for i, k in enumerate(pathway_keys)}

    commot_agg_list = []
    for slice_index, (adata_cur, data_cur, commot_path) in enumerate(
        zip(processed_transfer.adata_list, processed_transfer.spidernet_data, matched_commot_paths)
    ):
        commot_adata = sc.read_h5ad(input_path(commot_path))
        obs_index = pd.Index(adata_cur.obs.index.astype(str))
        commot_obs_index = pd.Index(commot_adata.obs_names.astype(str))
        missing_barcodes = obs_index.difference(commot_obs_index)
        if len(missing_barcodes) > 0:
            raise KeyError(
                f"{dataset_label}: {len(missing_barcodes)} cells are missing from COMMOT file {commot_path.name}. "
                f"First few missing barcodes: {missing_barcodes[:5].tolist()}"
            )

        edge_index_cur = data_cur["edge_index"]
        edge_np = edge_index_cur.detach().cpu().numpy()
        obs_pos = pd.Series(np.arange(commot_adata.n_obs), index=commot_obs_index)
        rows = obs_pos.loc[obs_index[edge_np[:, 0].astype(np.int64)]].to_numpy()
        cols = obs_pos.loc[obs_index[edge_np[:, 1].astype(np.int64)]].to_numpy()
        E = rows.shape[0]

        pathway_keys_cur = set(get_commot_pathway_keys(commot_adata))
        cellpair_pathway_array = np.zeros((E, len(pathway_keys)), dtype=np.float32)

        for pathway_key in pathway_keys:
            if pathway_key not in pathway_keys_cur:
                # Transfer-safe missing feature: keep zeros.
                continue
            pathway_matrix = commot_adata.obsp[pathway_key]
            j = pathway_to_col[pathway_key]
            if sp.issparse(pathway_matrix):
                pathway_matrix = pathway_matrix.tocsr()
                cellpair_pathway_array[:, j] = pathway_matrix[rows, cols].A1.astype(np.float32, copy=False)
            else:
                cellpair_pathway_array[:, j] = pathway_matrix[rows, cols].astype(np.float32, copy=False)

        cellpair_pathway_array = np.nan_to_num(cellpair_pathway_array, nan=0.0, posinf=0.0, neginf=0.0)

        num_cell_cur = data_cur.x.shape[0]
        edge_dst = edge_index_cur[:, 1].to(torch.int64).to(device)
        edge_src = edge_index_cur[:, 0].to(torch.int64).to(device)
        pathway_t = torch.tensor(cellpair_pathway_array, dtype=torch.float32, device=device)

        cell_pathway_receiver_agg = scatter_mean(
            pathway_t,
            edge_dst,
            dim=0,
            dim_size=num_cell_cur,
        ).to("cpu").numpy()

        cell_pathway_sender_agg = scatter_mean(
            pathway_t,
            edge_src,
            dim=0,
            dim_size=num_cell_cur,
        ).to("cpu").numpy()

        # Keep the historical COMMOT feature order used by existing regressors: receiver first, sender second.
        cell_pathway_agg_cur = np.hstack([cell_pathway_receiver_agg, cell_pathway_sender_agg])
        cell_pathway_agg_cur = np.nan_to_num(cell_pathway_agg_cur, nan=0.0, posinf=0.0, neginf=0.0)
        commot_agg_list.append(cell_pathway_agg_cur)

    commot_agg_all = np.vstack(commot_agg_list)
    return commot_agg_all, commot_feature_names


# Keep the old function name, but make it transfer-safe compatible.


# -----------------------------
# Regressors and transfer runner
# -----------------------------
def fit_or_load_training_regressors(
    processed_train,
    model,
    train_run_dir,
    device,
    nmflr_n_components=None,
    nmflr_random_state=0,
    nmflr_max_iter=1000,
):
    train_run_dir = Path(train_run_dir)
    reg_spider_path = train_run_dir / "reg_scale_dict.pkl"
    reg_nmflr_path = train_run_dir / "reg_scale_nmflr_dict.pkl"
    reg_commot_path = train_run_dir / "reg_scale_commot_dict.pkl"
    reg_banksy_path = train_run_dir / "reg_scale_banksy_dict.pkl"

    nmflr_model_path = train_run_dir / "NMF_LR_coronal_nmf_model.pkl"
    nmflr_colmax_path = train_run_dir / "NMF_LR_coronal_factor_colmax.pkl"
    banksy_pca_path = train_run_dir / "Banksy_coronal_pca_model.pkl"
    banksy_feature_names_path = train_run_dir / "Banksy_coronal_feature_names.pkl"
    commot_feature_names_path = train_run_dir / "COMMOT_coronal_feature_names.pkl"

    reg_scale_dict = load_pickle(input_path(reg_spider_path)) if input_path(reg_spider_path).exists() else None
    reg_scale_nmflr_dict = load_pickle(input_path(reg_nmflr_path)) if input_path(reg_nmflr_path).exists() else None
    reg_scale_commot_dict = load_pickle(input_path(reg_commot_path)) if input_path(reg_commot_path).exists() else None
    reg_scale_banksy_dict = load_pickle(input_path(reg_banksy_path)) if input_path(reg_banksy_path).exists() else None

    nmflr_model = load_pickle(input_path(nmflr_model_path)) if input_path(nmflr_model_path).exists() else None
    nmflr_colmax = load_pickle(input_path(nmflr_colmax_path)) if input_path(nmflr_colmax_path).exists() else None
    banksy_pca_model = load_pickle(input_path(banksy_pca_path)) if input_path(banksy_pca_path).exists() else None
    banksy_feature_names = load_pickle(input_path(banksy_feature_names_path)) if input_path(banksy_feature_names_path).exists() else None
    commot_feature_names = load_pickle(input_path(commot_feature_names_path)) if input_path(commot_feature_names_path).exists() else None

    # Backward compatibility: older cached NMF-LR models may not contain LR feature names.
    # Reconstruct/attach them from the current Coronal processed data when possible.
    if nmflr_model is not None and not hasattr(nmflr_model, "lr_feature_names_in_"):
        try:
            _, _, nmflr_feature_names_tmp = collect_lr_coexpression_matrix(
                processed_train.spidernet_data,
                return_feature_names=True,
            )
            if len(nmflr_feature_names_tmp) == int(getattr(nmflr_model, "n_features_in_")):
                nmflr_model.lr_feature_names_in_ = list(nmflr_feature_names_tmp)
                save_pickle(nmflr_model, output_path(nmflr_model_path))
                print(
                    "Attached reconstructed Coronal LR feature names to existing NMF-LR model "
                    f"and resaved: {nmflr_model_path}"
                )
            else:
                print(
                    "[NMF-LR warning] Reconstructed Coronal LR feature-name count does not match "
                    f"existing NMF model: {len(nmflr_feature_names_tmp)} vs "
                    f"{getattr(nmflr_model, 'n_features_in_', 'NA')}. Positional alignment will be used."
                )
        except Exception as e:
            print(
                "[NMF-LR warning] Could not reconstruct LR feature names for existing NMF model. "
                f"Positional alignment will be used. Error: {repr(e)}"
            )

    for label, obj, path in [
        ("SpiderNet regressors", reg_scale_dict, reg_spider_path),
        ("NMF-LR regressors", reg_scale_nmflr_dict, reg_nmflr_path),
        ("COMMOT regressors", reg_scale_commot_dict, reg_commot_path),
        ("Banksy regressors", reg_scale_banksy_dict, reg_banksy_path),
        ("Coronal NMF-LR model", nmflr_model, nmflr_model_path),
        ("Coronal Banksy PCA model", banksy_pca_model, banksy_pca_path),
        ("Coronal COMMOT feature names", commot_feature_names, commot_feature_names_path),
    ]:
        if obj is not None:
            print(f"Loaded {label} from: {path}")

    need_any_refit = any([
        reg_scale_dict is None,
        reg_scale_nmflr_dict is None,
        reg_scale_commot_dict is None,
        reg_scale_banksy_dict is None,
        nmflr_model is None,
        nmflr_colmax is None,
        banksy_pca_model is None,
        banksy_feature_names is None,
        commot_feature_names is None,
    ])

    if need_any_refit and not REFIT_REGRESSORS_IF_MISSING:
        raise FileNotFoundError(
            "One or more saved regressors / transfer models are missing, and "
            "REFIT_REGRESSORS_IF_MISSING is False."
        )

    celltype_all, _ = get_obs_concat(processed_train, ["celltype", "CELL_TYPE"], required=True)
    cellage_all, _ = get_obs_concat(processed_train, ["age", "SAMPLE_ID"], required=True)
    cellage_all = pd.to_numeric(cellage_all, errors="raise").astype(float)

    if reg_scale_dict is None:
        print("Refitting SpiderNet regressors from the training run outputs...")
        train_results = infer_meta_interactions(model=model, processed=processed_train)
        train_results = normalize_outputs(train_results)
        export_results(
            results=train_results,
            processed=processed_train,
            output_dir=output_path(train_run_dir),
        )
        mi_sr_agg_all = aggregate_mi_features(
            factor_envir_list=train_results["factor_envir_list"],
            spidernet_data_list=processed_train.spidernet_data,
            device=device,
        )
        reg_scale_dict = fit_celltype_regressors(mi_sr_agg_all, celltype_all, cellage_all)
        save_pickle(reg_scale_dict, output_path(reg_spider_path))
        print(f"Saved SpiderNet regressors to: {reg_spider_path}")

    if (reg_scale_nmflr_dict is None) or (nmflr_model is None) or (nmflr_colmax is None):
        print("Fitting Coronal NMF-LR model and regressors...")
        nmflr_factor_train_list, nmflr_model, nmflr_colmax = fit_training_nmflr_model(
            spidernet_data_list=processed_train.spidernet_data,
            n_components=nmflr_n_components,
            random_state=nmflr_random_state,
            max_iter=nmflr_max_iter,
        )
        nmflr_features_train = aggregate_nmflr_features(
            factor_lr_list=nmflr_factor_train_list,
            spidernet_data_list=processed_train.spidernet_data,
            device=device,
        )
        reg_scale_nmflr_dict = fit_celltype_regressors(nmflr_features_train, celltype_all, cellage_all)
        save_pickle(reg_scale_nmflr_dict, output_path(reg_nmflr_path))
        save_pickle(nmflr_model, output_path(nmflr_model_path))
        save_pickle(nmflr_colmax, output_path(nmflr_colmax_path))
        print(f"Saved NMF-LR regressors to: {reg_nmflr_path}")
        print(f"Saved Coronal NMF-LR model to: {nmflr_model_path}")

    train_feature_cfg = DATASET_FEATURE_CONFIG["Coronal"]

    if (reg_scale_commot_dict is None) or (commot_feature_names is None):
        print("Fitting COMMOT regressors from Coronal features and saving Coronal feature order...")
        commot_features_train, commot_feature_names = load_commot_aggregated_features_transfer_safe(
            processed_transfer=processed_train,
            dataset_label="Coronal",
            dataset_cfg=train_feature_cfg,
            allow_missing=False,
            reference_feature_names=None,
        )
        reg_scale_commot_dict = fit_celltype_regressors(commot_features_train, celltype_all, cellage_all)
        save_pickle(reg_scale_commot_dict, output_path(reg_commot_path))
        save_pickle(commot_feature_names, output_path(commot_feature_names_path))
        print(f"Saved COMMOT regressors to: {reg_commot_path}")
        print(f"Saved Coronal COMMOT feature names to: {commot_feature_names_path}")

    if (reg_scale_banksy_dict is None) or (banksy_pca_model is None) or (banksy_feature_names is None):
        print("Fitting Coronal Banksy PCA model and regressors...")
        banksy_features_train, banksy_pca_model, banksy_feature_names = fit_training_banksy_pca(
            processed_train=processed_train,
            dataset_cfg=train_feature_cfg,
            n_components=PCA_N_COMPONENTS,
        )
        reg_scale_banksy_dict = fit_celltype_regressors(banksy_features_train, celltype_all, cellage_all)
        save_pickle(reg_scale_banksy_dict, output_path(reg_banksy_path))
        save_pickle(banksy_pca_model, output_path(banksy_pca_path))
        save_pickle(banksy_feature_names, output_path(banksy_feature_names_path))
        print(f"Saved Banksy regressors to: {reg_banksy_path}")
        print(f"Saved Coronal Banksy PCA model to: {banksy_pca_path}")


    transfer_feature_artifacts = {
        "nmflr_model": nmflr_model,
        "nmflr_colmax": nmflr_colmax,
        "commot_feature_names": commot_feature_names,
        "banksy_pca_model": banksy_pca_model,
        "banksy_feature_names": banksy_feature_names,
    }

    return (
        reg_scale_dict,
        reg_scale_nmflr_dict,
        reg_scale_commot_dict,
        reg_scale_banksy_dict,
        transfer_feature_artifacts,
    )




### Enforce the coronal LR-pair space during transfer
External LR coexpression is rebuilt in the coronal training LR-pair order before SpiderNet inference and NMF-LR transformation. Missing ligand or receptor genes produce zero-valued columns while preserving the reference feature order.

In [ ]:
# =========================================================
# Coronal LR-pair enforcement for NMF-LR and SpiderNet inputs
# ---------------------------------------------------------
# External processed bundles may have been generated with dataset-specific LR
# pairs. For transfer-safe inference, external slices must use exactly the LR
# pairs selected in the Coronal training section. This cell rebuilds each
# external slice's edge-level LR coexpression matrix directly from the Coronal
# LR-pair list before SpiderNet inference and before NMF-LR transform.
#
# Important behavior:
#   * No positional padding/truncation is used.
#   * No external-specific LR-pair set is allowed.
#   * Missing ligand/receptor genes in an external slice produce a zero column
#     for that Coronal LR pair, but the LR-pair order remains identical.
# =========================================================

import ast
import re as _re


def _nmflr_is_positional_fallback_names(names):
    if names is None:
        return True
    names = [str(x) for x in names]
    return all(name == f"LR_feature_{i}" for i, name in enumerate(names))


def _nmflr_data_keys(data_cur):
    keys = []
    try:
        keys = list(data_cur.keys())
    except Exception:
        pass
    return [str(k) for k in keys]


def _nmflr_try_value_from_data(data_cur, key):
    try:
        if key in data_cur:
            return data_cur[key]
    except Exception:
        pass
    try:
        if hasattr(data_cur, key):
            return getattr(data_cur, key)
    except Exception:
        pass
    return None


def _nmflr_to_plain_list(value):
    if value is None:
        return None
    if torch.is_tensor(value):
        value = value.detach().cpu().numpy()
    if isinstance(value, pd.Index):
        value = value.tolist()
    if hasattr(value, "tolist") and not isinstance(value, (list, tuple)):
        value = value.tolist()
    if isinstance(value, (list, tuple, np.ndarray)):
        return list(value)
    return None


def _nmflr_parse_lr_pair_name(name):
    """Parse a stored LR-pair feature name into ligand and receptor strings."""
    name = str(name).strip()
    if len(name) == 0:
        return None, None

    # Tuple/list string, e.g. ('Lig', 'Rec')
    try:
        obj = ast.literal_eval(name)
        if isinstance(obj, (list, tuple)) and len(obj) >= 2:
            return str(obj[0]).strip(), str(obj[1]).strip()
    except Exception:
        pass

    # Remove common wrappers.
    cleaned = name
    cleaned = cleaned.replace("Ligand_Receptor", "")
    cleaned = cleaned.strip(" [](){}")

    # Prefer separators that are unlikely to appear inside gene symbols.
    separators = [
        "__", "___", "—", "--", "->", "→", "|", "::", ":", ";", " / ", "/", "_",
    ]
    for sep in separators:
        if sep in cleaned:
            parts = [p.strip() for p in cleaned.split(sep) if len(p.strip()) > 0]
            if len(parts) >= 2:
                return parts[0], parts[1]

    return None, None


def _nmflr_gene_group_to_string(gene_group):
    """Convert a SpiderNet LR-list ligand/receptor group into a '+'-joined string."""
    if gene_group is None:
        return ""
    if torch.is_tensor(gene_group):
        gene_group = gene_group.detach().cpu().numpy()
    if isinstance(gene_group, pd.Index):
        gene_group = gene_group.tolist()
    if hasattr(gene_group, "tolist") and not isinstance(gene_group, (list, tuple, str)):
        gene_group = gene_group.tolist()
    if isinstance(gene_group, (list, tuple, np.ndarray, set)):
        genes = [str(g).strip() for g in list(gene_group) if str(g).strip() not in {"", "nan", "None"}]
        return "+".join(genes)
    return str(gene_group).strip()


def _nmflr_reference_lr_pair_table_from_processed_lr_list(processed_train, n_features):
    """
    Recover the exact Coronal LR-pair order from processed_train.lr_list.

    In the SpiderNet processed bundle, cellpair_LRpair_neigh columns are ordered
    by processed_train.lr_list / LR_list.pkl. The PyG Data object may not store
    these names inside each data_cur, so this object-level attribute is the
    correct reference for transfer-safe NMF-LR rebuilding.
    """
    lr_list = getattr(processed_train, "lr_list", None)
    if lr_list is None:
        return None
    try:
        lr_list = list(lr_list)
    except Exception:
        return None

    if len(lr_list) != int(n_features):
        print(
            "[Coronal LR reference warning] processed_train.lr_list length does not match "
            f"cellpair_LRpair_neigh columns: len(lr_list)={len(lr_list)}, n_features={n_features}. "
            "Will try other LR-name sources."
        )
        return None

    rows = []
    failed_examples = []
    for idx, lr in enumerate(lr_list):
        ligand = receptor = None

        if isinstance(lr, (list, tuple)) and len(lr) >= 2:
            ligand = _nmflr_gene_group_to_string(lr[0])
            receptor = _nmflr_gene_group_to_string(lr[1])
        else:
            ligand, receptor = _nmflr_parse_lr_pair_name(str(lr))

        if ligand is None or receptor is None or len(str(ligand)) == 0 or len(str(receptor)) == 0:
            failed_examples.append(str(lr))
            continue

        rows.append({
            "feature_index": int(idx),
            "feature_name": f"{ligand}__{receptor}",
            "ligand": str(ligand),
            "receptor": str(receptor),
        })

    if len(rows) != int(n_features):
        raise ValueError(
            "processed_train.lr_list exists but could not be parsed into the full Coronal LR-pair table. "
            f"Parsed {len(rows)} / {n_features}. Failed examples: {failed_examples[:10]}"
        )

    ref_df = pd.DataFrame(rows)
    print(
        "Using processed_train.lr_list as the strict Coronal LR-pair reference "
        f"for transfer: n_features={ref_df.shape[0]}"
    )
    return ref_df


def _nmflr_get_reference_lr_pair_table_from_training(processed_train):
    """Return Coronal LR reference table with feature_name, ligand, receptor."""
    if len(processed_train.spidernet_data) == 0:
        raise ValueError("processed_train.spidernet_data is empty.")

    data_ref = processed_train.spidernet_data[0]
    if "cellpair_LRpair_neigh" not in data_ref:
        raise KeyError("Training processed data has no 'cellpair_LRpair_neigh'.")

    X_ref = data_ref["cellpair_LRpair_neigh"]
    n_features = int(X_ref.shape[1])

    # Preferred source: processed_train.lr_list / LR_list.pkl.
    # This is the canonical Coronal LR-pair order used to build
    # cellpair_LRpair_neigh, even when individual PyG Data objects do not
    # carry LR-pair names.
    ref_from_lr_list = _nmflr_reference_lr_pair_table_from_processed_lr_list(
        processed_train,
        n_features,
    )
    if ref_from_lr_list is not None:
        return ref_from_lr_list

    # First try explicit ligand/receptor vectors.
    ligand_keys = [
        "ligands", "ligand_names", "Ligand", "Ligand_gene", "ligand_gene",
        "ligand_genes", "Ligand_genes", "source_gene", "sender_gene",
    ]
    receptor_keys = [
        "receptors", "receptor_names", "Receptor", "Receptor_gene", "receptor_gene",
        "receptor_genes", "Receptor_genes", "target_gene", "receiver_gene",
    ]

    for lk in ligand_keys:
        lig_value = _nmflr_try_value_from_data(data_ref, lk)
        lig = _nmflr_to_feature_name_list(lig_value, n_features)
        if lig is None:
            continue
        for rk in receptor_keys:
            rec_value = _nmflr_try_value_from_data(data_ref, rk)
            rec = _nmflr_to_feature_name_list(rec_value, n_features)
            if rec is not None:
                feature_names = _nmflr_get_lr_feature_names_from_data(data_ref, n_features)
                if _nmflr_is_positional_fallback_names(feature_names):
                    feature_names = [f"{l}__{r}" for l, r in zip(lig, rec)]
                ref_df = pd.DataFrame({
                    "feature_index": np.arange(n_features, dtype=int),
                    "feature_name": [str(x) for x in feature_names],
                    "ligand": [str(x) for x in lig],
                    "receptor": [str(x) for x in rec],
                })
                print(
                    "Using explicit Coronal ligand/receptor columns for LR reference: "
                    f"{lk}, {rk}; n_features={n_features}"
                )
                return ref_df

    # Next try a list of tuple pairs stored under a pair-like key.
    pair_keys = [
        "lr_pairs", "LR_pairs", "lr_pair_list", "LR_pair_list", "LRpair_list",
        "ligand_receptor_pairs", "Ligand_Receptor_pairs", "cellpair_LRpair_neigh_pairs",
    ]
    for pk in pair_keys:
        pair_value = _nmflr_try_value_from_data(data_ref, pk)
        pairs = _nmflr_to_plain_list(pair_value)
        if pairs is None or len(pairs) != n_features:
            continue
        lig = []
        rec = []
        ok = True
        for p in pairs:
            if isinstance(p, (list, tuple)) and len(p) >= 2:
                lig.append(str(p[0]).strip())
                rec.append(str(p[1]).strip())
            else:
                l, r = _nmflr_parse_lr_pair_name(str(p))
                if l is None or r is None:
                    ok = False
                    break
                lig.append(l)
                rec.append(r)
        if ok:
            feature_names = [f"{l}__{r}" for l, r in zip(lig, rec)]
            ref_df = pd.DataFrame({
                "feature_index": np.arange(n_features, dtype=int),
                "feature_name": feature_names,
                "ligand": lig,
                "receptor": rec,
            })
            print(f"Using Coronal LR pair list key for LR reference: {pk}; n_features={n_features}")
            return ref_df

    # Last resort: parse feature names.
    feature_names = _nmflr_get_lr_feature_names_from_data(data_ref, n_features)
    if _nmflr_is_positional_fallback_names(feature_names):
        raise ValueError(
            "Could not recover Coronal LR-pair names from processed_train.spidernet_data. "
            "External transfer must use the Coronal-selected LR pairs, but only positional "
            "fallback names like LR_feature_0 were found. Please ensure the processed bundle "
            "stores ligand/receptor names or LR pair names. Available data keys: "
            f"{_nmflr_data_keys(data_ref)}"
        )

    parsed_rows = []
    failed = []
    for idx, name in enumerate(feature_names):
        ligand, receptor = _nmflr_parse_lr_pair_name(name)
        if ligand is None or receptor is None:
            failed.append(name)
        else:
            parsed_rows.append({
                "feature_index": int(idx),
                "feature_name": str(name),
                "ligand": ligand,
                "receptor": receptor,
            })

    if len(failed) > 0:
        raise ValueError(
            "Could not parse ligand/receptor from some Coronal LR feature names. "
            f"Failed examples: {failed[:10]}. Available data keys: {_nmflr_data_keys(data_ref)}"
        )

    ref_df = pd.DataFrame(parsed_rows)
    print(f"Parsed Coronal LR-pair names from feature names; n_features={ref_df.shape[0]}")
    return ref_df


def _split_gene_group_for_lr(expr):
    expr = str(expr).strip()
    expr = expr.strip(" [](){}")
    expr = expr.replace(";", "+").replace(",", "+").replace("&", "+").replace("/", "+").replace("|", "+")
    genes = [g.strip() for g in expr.split("+") if len(g.strip()) > 0]
    return genes if len(genes) > 0 else [expr]


def _get_gene_group_expression_from_adata(adata, gene_expr_cache, gene_group):
    genes = _split_gene_group_for_lr(gene_group)
    values = []
    var_upper_to_actual = gene_expr_cache["var_upper_to_actual"]

    for gene in genes:
        if gene in gene_expr_cache["gene_vectors"]:
            values.append(gene_expr_cache["gene_vectors"][gene])
            continue
        actual = gene if gene in gene_expr_cache["var_names"] else var_upper_to_actual.get(gene.upper(), None)
        if actual is None:
            gene_expr_cache["gene_vectors"][gene] = None
            continue
        idx = int(adata.var_names.get_loc(actual))
        x = adata.X[:, idx]
        if sp.issparse(x):
            x = x.toarray().ravel()
        else:
            x = np.asarray(x).ravel()
        x = np.asarray(x, dtype=np.float32)
        gene_expr_cache["gene_vectors"][gene] = x
        values.append(x)

    values = [v for v in values if v is not None]
    if len(values) == 0:
        return np.zeros(adata.n_obs, dtype=np.float32)
    if len(values) == 1:
        return values[0]
    return np.nanmean(np.vstack(values), axis=0).astype(np.float32, copy=False)


def _get_edge_index_numpy_for_lr(data_cur):
    edge_index = data_cur["edge_index"]
    if torch.is_tensor(edge_index):
        edge_index = edge_index.detach().cpu().numpy()
    else:
        edge_index = np.asarray(edge_index)
    if edge_index.ndim != 2:
        raise ValueError(f"edge_index must be 2D, got {edge_index.shape}")
    if edge_index.shape[0] == 2 and edge_index.shape[1] != 2:
        edge_index = edge_index.T
    if edge_index.shape[1] != 2:
        raise ValueError(f"edge_index must have two columns after conversion, got {edge_index.shape}")
    return edge_index.astype(int, copy=False)


def _build_reference_lr_coexpression_for_slice(adata, data_cur, reference_lr_df):
    edge_index = _get_edge_index_numpy_for_lr(data_cur)
    sender_idx = edge_index[:, 0].astype(int)
    receiver_idx = edge_index[:, 1].astype(int)

    var_names = pd.Index([str(x) for x in adata.var_names])
    gene_expr_cache = {
        "var_names": set(var_names),
        "var_upper_to_actual": {str(g).upper(): str(g) for g in var_names},
        "gene_vectors": {},
    }

    n_edges = edge_index.shape[0]
    n_features = reference_lr_df.shape[0]
    X_lr = np.zeros((n_edges, n_features), dtype=np.float32)

    missing_ligand = 0
    missing_receptor = 0
    for j, row in reference_lr_df.reset_index(drop=True).iterrows():
        ligand_expr = _get_gene_group_expression_from_adata(adata, gene_expr_cache, row["ligand"])
        receptor_expr = _get_gene_group_expression_from_adata(adata, gene_expr_cache, row["receptor"])
        if np.all(ligand_expr == 0):
            missing_ligand += 1
        if np.all(receptor_expr == 0):
            missing_receptor += 1
        X_lr[:, j] = ligand_expr[sender_idx] * receptor_expr[receiver_idx]

    X_lr = np.nan_to_num(X_lr, nan=0.0, posinf=0.0, neginf=0.0)
    X_lr[X_lr < 0] = 0.0
    return X_lr, missing_ligand, missing_receptor


def enforce_coronal_lr_pairs_for_transfer(processed_transfer, processed_train, output_dir=None, dataset_label="Transfer"):
    """
    Replace each transfer slice's cellpair_LRpair_neigh with coexpression
    computed from the Coronal training LR-pair list.
    """
    reference_lr_df = _nmflr_get_reference_lr_pair_table_from_training(processed_train)
    reference_names = reference_lr_df["feature_name"].astype(str).tolist()
    n_ref = len(reference_names)

    if output_dir is not None:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        reference_lr_df.to_csv(output_path(output_dir / f"{dataset_label}_Coronal_reference_LR_pairs_used_for_transfer.csv"), index=False)

    for slice_index, (adata_cur, data_cur) in enumerate(zip(processed_transfer.adata_list, processed_transfer.spidernet_data)):
        X_lr, missing_ligand, missing_receptor = _build_reference_lr_coexpression_for_slice(
            adata_cur,
            data_cur,
            reference_lr_df,
        )
        if X_lr.shape[1] != n_ref:
            raise ValueError(f"Internal error: rebuilt LR matrix has {X_lr.shape[1]} columns, expected {n_ref}.")

        data_cur["cellpair_LRpair_neigh"] = torch.tensor(X_lr, dtype=torch.float32)
        data_cur["cellpair_LRpair_neigh_names"] = reference_names
        data_cur["ligand_names"] = reference_lr_df["ligand"].astype(str).tolist()
        data_cur["receptor_names"] = reference_lr_df["receptor"].astype(str).tolist()

        print(
            f"{dataset_label} slice {slice_index}: rebuilt LR coexpression using Coronal LR pairs: "
            f"{X_lr.shape}; zero-ligand-pair-count={missing_ligand}; "
            f"zero-receptor-pair-count={missing_receptor}"
        )

    return processed_transfer


# -----------------------------
# Strict collect/transform: no padding/truncation
# -----------------------------
def collect_lr_coexpression_matrix(
    spidernet_data_list,
    reference_feature_names=None,
    reference_n_features=None,
    return_feature_names=False,
):
    """Return concatenated edge-level LR coexpression matrix with strict LR feature order."""
    lr_mats = []
    edge_counts = []
    reference_names = None if reference_feature_names is None else [str(x) for x in reference_feature_names]

    for slice_index, data_cur in enumerate(spidernet_data_list):
        if "cellpair_LRpair_neigh" not in data_cur:
            raise KeyError(
                "processed.spidernet_data contains no 'cellpair_LRpair_neigh'. "
                "NMF-LR baseline requires edge-level LR coexpression features."
            )
        X_cur = _clean_nonnegative_matrix_for_nmf(data_cur["cellpair_LRpair_neigh"])
        feature_names_cur = _nmflr_get_lr_feature_names_from_data(data_cur, X_cur.shape[1])
        feature_names_cur = [str(x) for x in feature_names_cur]

        if reference_names is None:
            reference_names = list(feature_names_cur)
            if reference_n_features is not None and len(reference_names) != int(reference_n_features):
                raise ValueError(
                    f"First LR feature set has {len(reference_names)} features, expected {reference_n_features}."
                )
        else:
            if X_cur.shape[1] != len(reference_names):
                raise ValueError(
                    f"LR feature-count mismatch in slice {slice_index}: current={X_cur.shape[1]}, "
                    f"reference={len(reference_names)}. External processed data must be rebuilt with "
                    "the Coronal LR-pair list before transfer."
                )
            if feature_names_cur != reference_names:
                mismatch = [
                    (j, feature_names_cur[j], reference_names[j])
                    for j in range(min(len(feature_names_cur), len(reference_names)))
                    if feature_names_cur[j] != reference_names[j]
                ]
                raise ValueError(
                    f"LR feature-order/name mismatch in slice {slice_index}. Examples: {mismatch[:10]}. "
                    "External processed data must use exactly the Coronal LR-pair order."
                )

        lr_mats.append(X_cur)
        edge_counts.append(X_cur.shape[0])
        print(
            f"Collected LR coexpression from slice {slice_index + 1}/{len(spidernet_data_list)}: "
            f"{X_cur.shape}"
        )

    if any(sp.issparse(X) for X in lr_mats):
        lr_all = sp.vstack(lr_mats, format="csr")
    else:
        lr_all = np.vstack(lr_mats)

    if return_feature_names:
        return lr_all, edge_counts, list(reference_names)
    return lr_all, edge_counts


def transform_nmflr_with_training_model(spidernet_data_list, nmf_lr_model, factor_colmax):
    """Transform external LR coexpression using the Coronal-fitted NMF-LR model.

    The transfer data must already have been rebuilt using the Coronal LR-pair
    list. This function intentionally performs no padding or feature imputation.
    """
    reference_feature_names = getattr(nmf_lr_model, "lr_feature_names_in_", None)
    reference_n_features = int(getattr(nmf_lr_model, "n_features_in_", 0)) or None

    if reference_feature_names is None:
        raise ValueError(
            "The Coronal-fitted NMF-LR model does not store lr_feature_names_in_. "
            "Please refit the Coronal NMF-LR model after running the strict LR-pair override."
        )

    lr_all, edge_counts = collect_lr_coexpression_matrix(
        spidernet_data_list,
        reference_feature_names=reference_feature_names,
        reference_n_features=reference_n_features,
    )
    print("Transforming external LR coexpression using Coronal-fitted NMF-LR model:", lr_all.shape)

    if reference_n_features is not None and lr_all.shape[1] != reference_n_features:
        raise ValueError(
            f"NMF-LR feature mismatch: external matrix has {lr_all.shape[1]} features, "
            f"but Coronal NMF expects {reference_n_features}."
        )

    factor_all = nmf_lr_model.transform(lr_all)
    factor_colmax = np.asarray(factor_colmax, dtype=float)
    factor_colmax[(~np.isfinite(factor_colmax)) | (factor_colmax == 0)] = 1.0
    factor_all = (factor_all / factor_colmax).astype(np.float32, copy=False)

    factor_lr_list = split_edge_factor_matrix(factor_all, edge_counts)
    return factor_lr_list


# -----------------------------
# Override transfer runner to force Coronal LR pairs before inference
# -----------------------------
def run_transfer_analysis(
    dataset_label,
    preprocess_fn,
    paths_obj,
    run_dirs_obj,
    processed_train,
    model,
    reg_scale_dict,
    reg_scale_nmflr_dict,
    reg_scale_commot_dict,
    reg_scale_banksy_dict,
    transfer_feature_artifacts,
    preprocess_cfg,
    old_age_threshold=19,
    plot_ymin=None,
    ratio_max=1.5,
    reference_processed_data_dir=TRAIN_PROCESSED_DATA_DIR,
):
    preprocess_fn()

    processed_transfer = load_processed_data(run_dirs_obj["run_dir"])
    print(f"{dataset_label}: loaded {len(processed_transfer.adata_list)} batches.")
    print(f"{dataset_label}: total cells = {np.sum([adata.n_obs for adata in processed_transfer.adata_list])}")

    # Critical: external LR features are rebuilt from the Coronal LR-pair list.
    processed_transfer = enforce_coronal_lr_pairs_for_transfer(
        processed_transfer=processed_transfer,
        processed_train=processed_train,
        output_dir=run_dirs_obj["run_dir"],
        dataset_label=dataset_label,
    )

    # Ensure the Coronal-fitted NMF-LR model uses the same explicit
    # Coronal LR-pair names. Existing cached models may have been fitted
    # before LR names were attached and may still carry positional names.
    reference_lr_df_for_nmflr = _nmflr_get_reference_lr_pair_table_from_training(processed_train)
    reference_lr_names_for_nmflr = reference_lr_df_for_nmflr["feature_name"].astype(str).tolist()
    nmflr_model_for_transfer = transfer_feature_artifacts.get("nmflr_model", None)
    if nmflr_model_for_transfer is not None:
        expected_n = int(getattr(nmflr_model_for_transfer, "n_features_in_", len(reference_lr_names_for_nmflr)))
        if expected_n != len(reference_lr_names_for_nmflr):
            raise ValueError(
                "Coronal NMF-LR model feature count does not match processed_train.lr_list: "
                f"model expects {expected_n}, but Coronal LR list has {len(reference_lr_names_for_nmflr)}."
            )
        nmflr_model_for_transfer.lr_feature_names_in_ = reference_lr_names_for_nmflr
        print(
            "Attached strict Coronal LR-pair names from processed_train.lr_list "
            "to the Coronal-fitted NMF-LR model for transfer."
        )

    processed_transfer, transfer_results = infer_transfer_results(
        model=model,
        processed_transfer=processed_transfer,
        processed_train=processed_train,
        output_dir=run_dirs_obj["run_dir"],
        reference_processed_data_dir=reference_processed_data_dir,
    )

    dataset_cfg = DATASET_FEATURE_CONFIG[dataset_label]

    mi_sr_agg_all = aggregate_mi_features(
        factor_envir_list=transfer_results["factor_envir_list"],
        spidernet_data_list=processed_transfer.spidernet_data,
        device=device,
    )

    transfer_nmflr_factor_list = transform_nmflr_with_training_model(
        spidernet_data_list=processed_transfer.spidernet_data,
        nmf_lr_model=transfer_feature_artifacts["nmflr_model"],
        factor_colmax=transfer_feature_artifacts["nmflr_colmax"],
    )
    transfer_nmflr_features = aggregate_nmflr_features(
        factor_lr_list=transfer_nmflr_factor_list,
        spidernet_data_list=processed_transfer.spidernet_data,
        device=device,
    )

    transfer_banksy_features = transform_banksy_with_training_pca(
        processed_transfer=processed_transfer,
        dataset_label=dataset_label,
        dataset_cfg=dataset_cfg,
        banksy_pca_model=transfer_feature_artifacts["banksy_pca_model"],
        banksy_feature_names=transfer_feature_artifacts["banksy_feature_names"],
    )

    transfer_commot_features = None
    if dataset_cfg.get("has_commot", False):
        transfer_commot_features, _ = load_commot_aggregated_features_transfer_safe(
            processed_transfer=processed_transfer,
            dataset_label=dataset_label,
            dataset_cfg=dataset_cfg,
            allow_missing=False,
            reference_feature_names=transfer_feature_artifacts["commot_feature_names"],
        )
    else:
        print(f"{dataset_label}: COMMOT benchmark is skipped because no COMMOT results are available for this dataset.")

    prediction_dict = {}
    ratio_dict = {}

    method_feature_regressor_pairs = [
        ("SpiderNet", mi_sr_agg_all, reg_scale_dict),
        ("NMF-LR", transfer_nmflr_features, reg_scale_nmflr_dict),
        ("Banksy", transfer_banksy_features, reg_scale_banksy_dict),
    ]

    if transfer_commot_features is not None:
        method_feature_regressor_pairs.append(("COMMOT", transfer_commot_features, reg_scale_commot_dict))

    for method_name, feature_matrix, reg_dict in method_feature_regressor_pairs:
        prediction_dict[method_name] = predict_age_from_features(
            feature_matrix=feature_matrix,
            processed_transfer=processed_transfer,
            regressors=reg_dict,
            method_name=method_name,
            dataset_label=dataset_label,
            output_dir=run_dirs_obj["run_dir"],
            old_age_threshold=old_age_threshold,
        )
        ratio_dict[method_name] = compute_old_young_ratio(
            prediction_dict[method_name],
            run_dirs_obj["run_dir"],
            dataset_label,
            method_name,
        )

    ratio_df_combined = combine_ratio_tables(ratio_dict)

    ratio_compare_path = Path(run_dirs_obj["run_dir"]) / f"{dataset_label}_Old_vs_Young_ratio_combined.csv"
    ratio_df_combined.to_csv(output_path(ratio_compare_path), index=False)
    print(f"Saved combined ratio table to: {ratio_compare_path}")

    plot_ratio_comparison(
        ratio_df_combined=ratio_df_combined,
        dataset_label=dataset_label,
        output_dir=run_dirs_obj["run_dir"],
        y_min=plot_ymin,
        y_max=ratio_max,
    )

    return {
        "processed_transfer": processed_transfer,
        "transfer_results": transfer_results,
        "mi_sr_agg_all": mi_sr_agg_all,
        "transfer_nmflr_features": transfer_nmflr_features,
        "transfer_nmflr_factor_list": transfer_nmflr_factor_list,
        "transfer_banksy_features": transfer_banksy_features,
        "transfer_commot_features": transfer_commot_features,
        "predictions": prediction_dict,
        "ratio_tables": ratio_dict,
        "pred_spider": prediction_dict.get("SpiderNet"),
        "pred_nmflr": prediction_dict.get("NMF-LR"),
        "pred_banksy": prediction_dict.get("Banksy"),
        "pred_commot": prediction_dict.get("COMMOT"),
        "spider_ratio": ratio_dict.get("SpiderNet"),
        "nmflr_ratio": ratio_dict.get("NMF-LR"),
        "banksy_ratio": ratio_dict.get("Banksy"),
        "commot_ratio": ratio_dict.get("COMMOT"),
        "ratio_df_combined": ratio_df_combined,
    }


## 5. Load or refit the training age-prediction regressors

### Baseline feature spaces

The transfer analysis uses coronal-fitted mappings for every baseline that requires a learned basis or feature order. The active baselines are NMF-LR, COMMOT, and Banksy; the gene-expression PCA implementation retained in the helper cell is not executed by the active workflow.


In [ ]:

(
    reg_scale_dict,
    reg_scale_nmflr_dict,
    reg_scale_commot_dict,
    reg_scale_banksy_dict,
    transfer_feature_artifacts,
) = fit_or_load_training_regressors(
    processed_train=processed_train,
    model=model,
    train_run_dir=train_run_dirs["run_dir"],
    device=device,
    nmflr_n_components=NMF_LR_N_COMPONENTS,
    nmflr_random_state=NMF_LR_RANDOM_STATE,
    nmflr_max_iter=NMF_LR_MAX_ITER,
)

print("Number of SpiderNet regressors:", len(reg_scale_dict))
print("Number of NMF-LR regressors:", len(reg_scale_nmflr_dict))
print("Number of COMMOT regressors:", len(reg_scale_commot_dict))
print("Number of Banksy regressors:", len(reg_scale_banksy_dict))
print("Transfer-safe feature artifacts:", sorted(transfer_feature_artifacts.keys()))



## 6. Transfer to sagittal slices

This section loads the preprocessed sagittal bundle, enforces the coronal LR-pair feature space, infers MIs with the trained SpiderNet model, transforms the available baseline features, and summarizes old-to-young mean predicted-age ratios by cell type. COMMOT is skipped because no sagittal COMMOT result is configured.


In [ ]:
sagittal_results = run_transfer_analysis(
    dataset_label="Sagittal",
    preprocess_fn=lambda: preprocess_sagittal_dataset(
        processed_dir=SAGITTAL_PROCESSED_DATA_DIR,
        run_dirs_obj=sagittal_run_dirs,
    ),
    paths_obj=sagittal_paths,
    run_dirs_obj=sagittal_run_dirs,
    processed_train=processed_train,
    model=model,
    reg_scale_dict=reg_scale_dict,
    reg_scale_nmflr_dict=reg_scale_nmflr_dict,
    reg_scale_commot_dict=reg_scale_commot_dict,
    reg_scale_banksy_dict=reg_scale_banksy_dict,
    transfer_feature_artifacts=transfer_feature_artifacts,
    preprocess_cfg=preprocess_cfg,
    old_age_threshold=OLD_AGE_THRESHOLD,
    plot_ymin=0.4,
    ratio_max=1.4,
    reference_processed_data_dir=TRAIN_PROCESSED_DATA_DIR
)


In [ ]:
sagittal_results['ratio_df_combined']


## 7. Transfer to hippocampus

This section applies the same coronal-reference transfer workflow to the preprocessed hippocampus Stereo-seq bundle. The source-study sample labels determine the old and young groups when numeric ages are unavailable.


In [ ]:
hippocampus_results = run_transfer_analysis(
    dataset_label="Hippocampus",
    preprocess_fn=lambda: preprocess_hippocampus_dataset(
        processed_dir=HIPPOCAMPUS_PROCESSED_DATA_DIR,
        run_dirs_obj=hippocampus_run_dirs,
    ),
    paths_obj=hippocampus_paths,
    run_dirs_obj=hippocampus_run_dirs,
    processed_train=processed_train,
    model=model,
    reg_scale_dict=reg_scale_dict,
    reg_scale_nmflr_dict=reg_scale_nmflr_dict,
    reg_scale_commot_dict=reg_scale_commot_dict,
    reg_scale_banksy_dict=reg_scale_banksy_dict,
    transfer_feature_artifacts=transfer_feature_artifacts,
    preprocess_cfg=preprocess_cfg,
    old_age_threshold=OLD_AGE_THRESHOLD,
    plot_ymin=0.35,
    ratio_max=2,
    reference_processed_data_dir=TRAIN_PROCESSED_DATA_DIR
)


In [ ]:
hippocampus_results['ratio_df_combined']

## 8. Combined summary table

In [ ]:
combined_ratio_summary = pd.concat([
    sagittal_results["ratio_df_combined"].assign(Dataset="Sagittal"),
    hippocampus_results["ratio_df_combined"].assign(Dataset="Hippocampus"),
], axis=0, ignore_index=True)

preferred_columns = [
    "Dataset",
    "CellType",
    "SpiderNet_Old_vs_Young_ratio",
    "NMF-LR_Old_vs_Young_ratio",
    "COMMOT_Old_vs_Young_ratio",
    "Banksy_Old_vs_Young_ratio",
]
combined_ratio_summary = combined_ratio_summary[
    [col for col in preferred_columns if col in combined_ratio_summary.columns]
]

combined_ratio_summary.to_csv(
    output_path(train_run_dirs["run_dir"] / "Transfer_old_vs_young_ratio_summary.csv"),
    index=False
)

combined_ratio_summary

In [ ]:
# ==============================================================
# Sagittal edge-level T cell-to-neighbor MI-29 analysis
# --------------------------------------------------------------
# Compare older vs younger sagittal groups using every directed
# T cell -> neighboring-cell edge as one observation.
# No sample-level / slice-level mean aggregation is applied here.
#
# Plot style:
#   Use the same visual encoding as the downstream ageing-score comparison:
#     - order: Older, Younger
#     - Older uses High color
#     - Younger uses Low color
#     - white median line
#     - no cap lines
#
# Required upstream object:
#   sagittal_results, generated by run_transfer_analysis(..., dataset_label="Sagittal")
# ==============================================================

from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu


SAGITTAL_MI_OI = "MI29"
SAGITTAL_MI_INDEX = int(SAGITTAL_MI_OI.replace("MI", "").replace("-", "")) - 1
SAGITTAL_SENDER_CELLTYPE = "T cell"

# Display older samples first and retain the established high/low palette.
SAGITTAL_AGE_GROUP_ORDER = ["Older", "Younger"]
SAGITTAL_AGE_GROUP_PALETTE = {
    "Older": "#e93732",    # High color
    "Younger": "#d9adac",  # Low color
}


def _sagittal_get_edge_index_numpy(data_cur):
    edge_index = data_cur["edge_index"]
    if hasattr(edge_index, "detach"):
        edge_index = edge_index.detach().cpu().numpy()
    else:
        edge_index = np.asarray(edge_index)

    if edge_index.ndim != 2:
        raise ValueError(f"edge_index must be 2-dimensional, got shape {edge_index.shape}")

    # Support either [n_edges, 2] or PyG-style [2, n_edges].
    if edge_index.shape[0] == 2 and edge_index.shape[1] != 2:
        edge_index = edge_index.T

    if edge_index.shape[1] != 2:
        raise ValueError(f"edge_index must have two columns after conversion, got shape {edge_index.shape}")

    return edge_index.astype(int, copy=False)


def _sagittal_get_factor_numpy(factor_cur):
    if hasattr(factor_cur, "detach"):
        factor_cur = factor_cur.detach().cpu().numpy()
    else:
        factor_cur = np.asarray(factor_cur)

    if factor_cur.ndim != 2:
        raise ValueError(f"factor_envir must be 2-dimensional, got shape {factor_cur.shape}")

    return factor_cur.astype(float, copy=False)


def _sagittal_first_nonempty_obs_value(adata, columns, default=np.nan):
    for col in columns:
        if col is None or col not in adata.obs.columns:
            continue
        vals = pd.Series(adata.obs[col]).dropna().astype(str).str.strip()
        vals = vals[(vals != "") & (vals.str.lower() != "nan")]
        if len(vals) > 0:
            return vals.iloc[0]
    return default


def _sagittal_extract_first_number(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"[-+]?\d*\.?\d+", str(value))
    if match is None:
        return np.nan
    try:
        return float(match.group(0))
    except Exception:
        return np.nan


def _sagittal_assign_age_group(age_value, old_age_threshold=None):
    age_numeric = _sagittal_extract_first_number(age_value)
    if old_age_threshold is None:
        old_age_threshold = globals().get("OLD_AGE_THRESHOLD", 19)

    if np.isfinite(age_numeric):
        return ("Older" if age_numeric >= old_age_threshold else "Younger"), age_numeric

    age_text = str(age_value).lower()
    if "old" in age_text or "older" in age_text or "aged" in age_text:
        return "Older", np.nan
    if "young" in age_text or "younger" in age_text:
        return "Younger", np.nan

    return "Unknown", np.nan


def _sagittal_compute_smd(group_a_vals, group_b_vals):
    """Standardized mean difference: group_a - group_b."""
    group_a_vals = pd.to_numeric(pd.Series(group_a_vals), errors="coerce").dropna().to_numpy(dtype=float)
    group_b_vals = pd.to_numeric(pd.Series(group_b_vals), errors="coerce").dropna().to_numpy(dtype=float)

    if len(group_a_vals) == 0 or len(group_b_vals) == 0:
        return np.nan

    if len(group_a_vals) > 1 and len(group_b_vals) > 1:
        pooled_var = (
            (len(group_a_vals) - 1) * np.var(group_a_vals, ddof=1)
            + (len(group_b_vals) - 1) * np.var(group_b_vals, ddof=1)
        ) / (len(group_a_vals) + len(group_b_vals) - 2)
        pooled_sd = np.sqrt(pooled_var)
    else:
        pooled_sd = np.nan

    if not np.isfinite(pooled_sd) or pooled_sd == 0:
        return np.nan

    return (np.mean(group_a_vals) - np.mean(group_b_vals)) / pooled_sd


def _sagittal_p_label(p):
    if not np.isfinite(p):
        return "p = NA"
    if p < 1e-4:
        return "p < 1e-4"
    return f"p = {p:.2e}"


if "sagittal_results" not in globals():
    raise NameError("Please run the Sagittal transfer-analysis cell first so that sagittal_results exists.")

processed_sagittal = sagittal_results["processed_transfer"]
sagittal_transfer_results = sagittal_results["transfer_results"]
sagittal_outdir = Path(sagittal_run_dirs["run_dir"])
sagittal_outdir.mkdir(parents=True, exist_ok=True)

edge_df_parts = []

for batch_index, (adata_cur, data_cur, factor_cur) in enumerate(
    zip(
        processed_sagittal.adata_list,
        processed_sagittal.spidernet_data,
        sagittal_transfer_results["factor_envir_list"],
    )
):
    edge_index_cur = _sagittal_get_edge_index_numpy(data_cur)
    factor_envir_cur = _sagittal_get_factor_numpy(factor_cur)

    if SAGITTAL_MI_INDEX >= factor_envir_cur.shape[1]:
        raise IndexError(
            f"{SAGITTAL_MI_OI} requires column {SAGITTAL_MI_INDEX}, "
            f"but factor_envir has only {factor_envir_cur.shape[1]} columns."
        )

    celltype_all_cur = np.asarray(adata_cur.obs[CELL_TYPE_COL].astype(str))
    sender_idx_all = edge_index_cur[:, 0].astype(int)
    receiver_idx_all = edge_index_cur[:, 1].astype(int)
    sender_is_tcell = celltype_all_cur[sender_idx_all] == SAGITTAL_SENDER_CELLTYPE

    if np.sum(sender_is_tcell) == 0:
        continue

    sender_idx = sender_idx_all[sender_is_tcell]
    receiver_idx = receiver_idx_all[sender_is_tcell]
    mi29_edge_values = factor_envir_cur[sender_is_tcell, SAGITTAL_MI_INDEX].astype(float)

    age_value = _sagittal_first_nonempty_obs_value(
        adata_cur,
        ["age", SAMPLE_ID_COL, "sample_age", "Age"],
        default=np.nan,
    )
    age_group, age_numeric = _sagittal_assign_age_group(
        age_value,
        old_age_threshold=globals().get("OLD_AGE_THRESHOLD", 19),
    )
    sample_id = _sagittal_first_nonempty_obs_value(
        adata_cur,
        ["sample", "sample_name_full", "sample_name", "SAMPLE_ID", SAMPLE_ID_COL, "age"],
        default=f"Sagittal_slice_{batch_index}",
    )

    edge_df_cur = pd.DataFrame({
        "Dataset": "Sagittal",
        "slice_index": batch_index,
        "sample_id": sample_id,
        "sample_unit": f"{sample_id}__slice{batch_index}",
        "age_value": age_value,
        "age_numeric": age_numeric,
        "AgeGroup": age_group,
        "MI": SAGITTAL_MI_OI,
        "edge_index_in_slice": np.where(sender_is_tcell)[0].astype(int),
        "sender_index": sender_idx,
        "receiver_index": receiver_idx,
        "sender_barcode": np.asarray(adata_cur.obs_names.astype(str))[sender_idx],
        "receiver_barcode": np.asarray(adata_cur.obs_names.astype(str))[receiver_idx],
        "sender_celltype": celltype_all_cur[sender_idx],
        "receiver_celltype": celltype_all_cur[receiver_idx],
        "edge_Tcell_to_neighbor_MI29": mi29_edge_values,
    })
    edge_df_parts.append(edge_df_cur)

if len(edge_df_parts) == 0:
    raise ValueError("No directed T cell -> neighboring-cell edges were found in the Sagittal dataset.")

sagittal_edge_mi29_tcell_to_neighbor_df = pd.concat(edge_df_parts, axis=0, ignore_index=True)

edge_table_path = sagittal_outdir / "Sagittal_edge_level_Tcell_to_neighbor_MI29_strength.csv"
sagittal_edge_mi29_tcell_to_neighbor_df.to_csv(output_path(edge_table_path), index=False)
print(f"Saved edge-level MI-29 table to: {edge_table_path}")

score_numeric = pd.to_numeric(
    sagittal_edge_mi29_tcell_to_neighbor_df["edge_Tcell_to_neighbor_MI29"],
    errors="coerce",
)

plot_df = sagittal_edge_mi29_tcell_to_neighbor_df[
    sagittal_edge_mi29_tcell_to_neighbor_df["AgeGroup"].isin(SAGITTAL_AGE_GROUP_ORDER)
    & np.isfinite(score_numeric)
].copy()

plot_df["edge_Tcell_to_neighbor_MI29"] = pd.to_numeric(
    plot_df["edge_Tcell_to_neighbor_MI29"],
    errors="coerce",
)

older_vals = plot_df.loc[
    plot_df["AgeGroup"] == "Older",
    "edge_Tcell_to_neighbor_MI29",
]

younger_vals = plot_df.loc[
    plot_df["AgeGroup"] == "Younger",
    "edge_Tcell_to_neighbor_MI29",
]

if len(younger_vals) > 0 and len(older_vals) > 0:
    stat, pval = mannwhitneyu(younger_vals, older_vals, alternative="two-sided")
else:
    stat, pval = np.nan, np.nan

smd_older_minus_younger = _sagittal_compute_smd(older_vals, younger_vals)

sagittal_edge_mi29_stats = pd.DataFrame([{
    "Dataset": "Sagittal",
    "MI": SAGITTAL_MI_OI,
    "comparison": "Older_vs_Younger",
    "score": "edge_Tcell_to_neighbor_MI29",
    "analysis_level": "edge_level_Tcell_to_neighbor",
    "n_older_edges": int(len(older_vals)),
    "n_younger_edges": int(len(younger_vals)),
    "mean_older": float(np.nanmean(older_vals)) if len(older_vals) > 0 else np.nan,
    "mean_younger": float(np.nanmean(younger_vals)) if len(younger_vals) > 0 else np.nan,
    "median_older": float(np.nanmedian(older_vals)) if len(older_vals) > 0 else np.nan,
    "median_younger": float(np.nanmedian(younger_vals)) if len(younger_vals) > 0 else np.nan,
    "smd_older_minus_younger": smd_older_minus_younger,
    "mannwhitneyu_stat": stat,
    "pvalue": pval,
}])

stats_path = sagittal_outdir / "Sagittal_edge_level_Tcell_to_neighbor_MI29_old_vs_young_stats.csv"
sagittal_edge_mi29_stats.to_csv(output_path(stats_path), index=False)
print(f"Saved edge-level MI-29 old-vs-young stats to: {stats_path}")

display(sagittal_edge_mi29_tcell_to_neighbor_df.head())
display(sagittal_edge_mi29_stats)


# Boxplot: one observation per T cell -> neighboring-cell edge
# Use the established ageing-score boxplot style without overlaid points.
# -----------------------------
plt.figure(figsize=(3.1, 4.0))
ax = plt.gca()

sns.boxplot(
    data=plot_df,
    x="AgeGroup",
    y="edge_Tcell_to_neighbor_MI29",
    order=SAGITTAL_AGE_GROUP_ORDER,
    palette=SAGITTAL_AGE_GROUP_PALETTE,
    width=0.7,
    linewidth=1.4,
    showfliers=False,
    medianprops=dict(color="white", linewidth=1.8),
    capprops=dict(linewidth=0),
    ax=ax,
)

ax.set_xlabel("Sagittal age group", fontsize=14)
ax.set_ylabel("Edge-level T cell → neighboring-cell MI-29", fontsize=14)
ax.tick_params(axis="both", labelsize=12)

ax.text(
    0.5,
    0.98,
    f"{_sagittal_p_label(pval)}\nSMD = {smd_older_minus_younger:.2f}",
    ha="center",
    va="top",
    fontsize=11,
    transform=ax.transAxes,
)

sns.despine(top=True, right=True)
plt.tight_layout()

plot_path = sagittal_outdir / "Sagittal_edge_level_Tcell_to_neighbor_MI29_old_vs_young_boxplot.pdf"
plt.savefig(output_path(plot_path), dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print(f"Saved edge-level MI-29 boxplot to: {plot_path}")

In [ ]:
# ==============================================================
# Sagittal ageing-module score by T-cell MI-29 group
# --------------------------------------------------------------
# Apply the AgingBrain_training_analysis logic to the Sagittal dataset:
#   1. build Aging module score by global z-scored + mean expression,
#   2. sum-aggregate T cell -> neighboring-cell MI-29 for each T cell,
#   3. split T cells into Low/High MI-29 groups,
#   4. compare Aging module score in:
#      a) T cells themselves,
#      b) their neighboring receiver cells.
#
# Required upstream object:
#   sagittal_results, generated by run_transfer_analysis(..., dataset_label="Sagittal")
# ==============================================================

from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse
from scipy.stats import mannwhitneyu


SAGITTAL_MI_OI = "MI29"
SAGITTAL_MI_INDEX = int(SAGITTAL_MI_OI.replace("MI", "").replace("-", "")) - 1
SAGITTAL_SENDER_CELLTYPE = "T cell"
SAGITTAL_TCELL_MI29_GROUP_ORDER = ["High", "Low"]
SAGITTAL_TCELL_GROUP_PALETTE = {
    "High": "#3d91cf",
    "Low": "#83b7de",
}
SAGITTAL_NEIGHBOR_GROUP_PALETTE = {
    "High": "#e93732",
    "Low": "#d9adac",
}
SAGITTAL_MODULE_SCORE_METHOD = "zscore_mean_global"
SAGITTAL_TCELL_OUTGOING_AGG = "sum"


def _sagittal_get_edge_index_numpy(data_cur):
    edge_index = data_cur["edge_index"]
    if hasattr(edge_index, "detach"):
        edge_index = edge_index.detach().cpu().numpy()
    else:
        edge_index = np.asarray(edge_index)

    if edge_index.ndim != 2:
        raise ValueError(f"edge_index must be 2-dimensional, got shape {edge_index.shape}")

    if edge_index.shape[0] == 2 and edge_index.shape[1] != 2:
        edge_index = edge_index.T

    if edge_index.shape[1] != 2:
        raise ValueError(f"edge_index must have two columns after conversion, got shape {edge_index.shape}")

    return edge_index.astype(int, copy=False)


def _sagittal_get_factor_numpy(factor_cur):
    if hasattr(factor_cur, "detach"):
        factor_cur = factor_cur.detach().cpu().numpy()
    else:
        factor_cur = np.asarray(factor_cur)
    if factor_cur.ndim != 2:
        raise ValueError(f"factor_envir must be 2-dimensional, got shape {factor_cur.shape}")
    return factor_cur.astype(float, copy=False)


def _sagittal_first_nonempty_obs_value(adata, columns, default=np.nan):
    for col in columns:
        if col is None or col not in adata.obs.columns:
            continue
        vals = pd.Series(adata.obs[col]).dropna().astype(str).str.strip()
        vals = vals[(vals != "") & (vals.str.lower() != "nan")]
        if len(vals) > 0:
            return vals.iloc[0]
    return default


def _sagittal_extract_first_number(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"[-+]?\d*\.?\d+", str(value))
    if match is None:
        return np.nan
    try:
        return float(match.group(0))
    except Exception:
        return np.nan


def _sagittal_assign_age_group(age_value, old_age_threshold=None):
    age_numeric = _sagittal_extract_first_number(age_value)
    if old_age_threshold is None:
        old_age_threshold = globals().get("OLD_AGE_THRESHOLD", 19)

    if np.isfinite(age_numeric):
        return ("Older" if age_numeric >= old_age_threshold else "Younger"), age_numeric

    age_text = str(age_value).lower()
    if "old" in age_text or "older" in age_text or "aged" in age_text:
        return "Older", np.nan
    if "young" in age_text or "younger" in age_text:
        return "Younger", np.nan

    return "Unknown", np.nan


def _sagittal_resolve_gene_names_from_varnames(var_names, genes):
    """Resolve requested gene symbols to actual var_names, case-insensitively."""
    var_names = pd.Index([str(x) for x in var_names])
    upper_to_actual = {}
    for g in var_names:
        upper_to_actual.setdefault(str(g).upper(), str(g))

    resolved = {}
    missing = []
    for gene in genes:
        gene = str(gene).strip()
        if len(gene) == 0:
            continue
        if gene in var_names:
            resolved[gene] = gene
        elif gene.upper() in upper_to_actual:
            resolved[gene] = upper_to_actual[gene.upper()]
        else:
            missing.append(gene)

    return resolved, missing


def _sagittal_get_dense_gene_matrix(adata, genes_actual):
    gene_idx = [adata.var_names.get_loc(g) for g in genes_actual]
    X_sub = adata.X[:, gene_idx]
    if sparse.issparse(X_sub):
        X_sub = X_sub.toarray()
    else:
        X_sub = np.asarray(X_sub)
    return X_sub.astype(float, copy=False)


def _sagittal_compute_smd(high_vals, low_vals):
    """Standardized mean difference: High - Low."""
    high_vals = pd.to_numeric(pd.Series(high_vals), errors="coerce").dropna().to_numpy(dtype=float)
    low_vals = pd.to_numeric(pd.Series(low_vals), errors="coerce").dropna().to_numpy(dtype=float)

    if len(high_vals) == 0 or len(low_vals) == 0:
        return np.nan

    if len(high_vals) > 1 and len(low_vals) > 1:
        pooled_var = (
            (len(high_vals) - 1) * np.var(high_vals, ddof=1)
            + (len(low_vals) - 1) * np.var(low_vals, ddof=1)
        ) / (len(high_vals) + len(low_vals) - 2)
        pooled_sd = np.sqrt(pooled_var)
    else:
        pooled_sd = np.nan

    if not np.isfinite(pooled_sd) or pooled_sd == 0:
        return np.nan

    return (np.mean(high_vals) - np.mean(low_vals)) / pooled_sd


def _sagittal_p_label(p):
    if not np.isfinite(p):
        return "p = NA"
    if p < 1e-4:
        return "p < 1e-4"
    return f"p = {p:.2e}"


def _sagittal_assign_high_low_by_quantile(df, value_col, group_col, q=2):
    df = df.copy()
    valid = pd.to_numeric(df[value_col], errors="coerce")
    valid_mask = np.isfinite(valid)

    df[group_col] = np.nan

    if valid_mask.sum() == 0:
        raise ValueError(f"No finite values found in {value_col}.")

    valid_values = valid.loc[valid_mask]
    if valid_values.nunique(dropna=True) < 2:
        raise ValueError(
            f"{value_col} has fewer than two unique finite values; cannot create Low/High groups."
        )

    try:
        assigned = pd.qcut(
            valid_values,
            q=q,
            labels=["Low", "High"],
            duplicates="drop",
        )
        if len(pd.unique(assigned.dropna())) < 2:
            raise ValueError("qcut produced fewer than two groups.")
        df.loc[valid_mask, group_col] = assigned.astype(str)
    except Exception:
        threshold = float(np.nanmedian(valid_values))
        df.loc[valid_mask, group_col] = np.where(valid_values > threshold, "High", "Low")

    df[group_col] = pd.Categorical(
        df[group_col],
        categories=SAGITTAL_TCELL_MI29_GROUP_ORDER,
        ordered=True,
    )
    return df

def _sagittal_plot_grouped_aging_score(
    data,
    score_col,
    output_prefix,
    y_label,
    x_label,
    color_map,
):
    plot_df = data[
        data["Tcell_MI29_group"].isin(SAGITTAL_TCELL_MI29_GROUP_ORDER)
        & np.isfinite(pd.to_numeric(data[score_col], errors="coerce"))
    ].copy()

    low_vals = pd.to_numeric(
        plot_df.loc[plot_df["Tcell_MI29_group"] == "Low", score_col],
        errors="coerce",
    ).dropna()
    high_vals = pd.to_numeric(
        plot_df.loc[plot_df["Tcell_MI29_group"] == "High", score_col],
        errors="coerce",
    ).dropna()

    if len(low_vals) > 0 and len(high_vals) > 0:
        stat, pval = mannwhitneyu(low_vals, high_vals, alternative="two-sided")
    else:
        stat, pval = np.nan, np.nan

    smd = _sagittal_compute_smd(high_vals, low_vals)

    stats_df = pd.DataFrame([{
        "Dataset": "Sagittal",
        "MI": SAGITTAL_MI_OI,
        "score": score_col,
        "group_high": "High",
        "group_low": "Low",
        "n_high": int(len(high_vals)),
        "n_low": int(len(low_vals)),
        "mean_high": float(np.nanmean(high_vals)) if len(high_vals) > 0 else np.nan,
        "mean_low": float(np.nanmean(low_vals)) if len(low_vals) > 0 else np.nan,
        "smd_high_minus_low": smd,
        "mannwhitneyu_stat": stat,
        "pvalue": pval,
    }])

    stats_path = sagittal_outdir / f"{output_prefix}_stats.csv"
    stats_df.to_csv(output_path(stats_path), index=False)

    display(stats_df)

    plt.figure(figsize=(3.1, 4.0))
    ax = plt.gca()

    sns.boxplot(
        data=plot_df,
        x="Tcell_MI29_group",
        y=score_col,
        order=SAGITTAL_TCELL_MI29_GROUP_ORDER,
        palette=color_map,
        width=0.7,
        linewidth=1.4,
        showfliers=False,
        medianprops=dict(color="white", linewidth=1.8),
        capprops=dict(linewidth=0),
        ax=ax,
    )

    ax.set_xlabel(x_label, fontsize=14)
    ax.set_ylabel(y_label, fontsize=14)
    ax.tick_params(axis="both", labelsize=12)

    ax.text(
        0.5,
        0.98,
        f"{_sagittal_p_label(pval)}\nSMD = {smd:.2f}",
        ha="center",
        va="top",
        fontsize=11,
        transform=ax.transAxes,
    )

    sns.despine(top=True, right=True)
    plt.tight_layout()

    plot_path = sagittal_outdir / f"{output_prefix}.pdf"
    plt.savefig(output_path(plot_path), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    print(f"Saved plot to: {plot_path}")
    print(f"Saved stats to: {stats_path}")

    return stats_df


if "sagittal_results" not in globals():
    raise NameError("Please run the Sagittal transfer-analysis cell first so that sagittal_results exists.")

processed_sagittal = sagittal_results["processed_transfer"]
sagittal_transfer_results = sagittal_results["transfer_results"]
sagittal_outdir = Path(sagittal_run_dirs["run_dir"])
sagittal_outdir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# 1. Select good aging genes from the Coronal training section,
#    then build Sagittal Aging module score using only these genes
# -----------------------------
# Good aging genes are selected from the Coronal section (processed_train):
#   1. build the aging/senescence gene set from the MERFISH gene panel,
#   2. sort Coronal slices by numeric age,
#   3. use the 5 youngest and 5 oldest Coronal slices,
#   4. compute cell-level log2FC(old vs young),
#   5. keep genes with log2FC_old_vs_young_cell_mean > 0.2.
#
# The Sagittal Aging module score below is then calculated by global
# z-scored + mean expression using only these Coronal-selected good genes.
# -----------------------------
GOOD_AGING_GENE_CELL_LFC_THRESHOLD = 0.2
CORONAL_GOOD_GENE_N_YOUNG_SLICES = 5
CORONAL_GOOD_GENE_N_OLD_SLICES = 5
CORONAL_GOOD_GENE_LFC_PSEUDOCOUNT = 1e-6
SAGITTAL_MODULE_SCORE_METHOD = "zscore_mean_global_coronal_good_aging_genes_cellLFC_gt0.2"


def _sagittal_get_numeric_slice_age_from_adata(adata, columns):
    age_value = _sagittal_first_nonempty_obs_value(adata, columns, default=np.nan)
    age_numeric = _sagittal_extract_first_number(age_value)
    return age_value, age_numeric


if "processed_train" not in globals():
    raise NameError(
        "processed_train is required to select good aging genes from the Coronal training section. "
        "Please run the training processed-data loading cell first."
    )

# -----------------------------
# 1A. Build aging/senescence gene set from the MERFISH gene panel
# -----------------------------
gene_panel_filename = "2023-12-22736D-TableS1_MERFISHGenePanel.xlsx"
gene_panel_candidates = [
    TRAIN_DATA_ROOT / "Supp_table" / gene_panel_filename,
    TRAIN_DATA_ROOT.parent / "Supp_table" / gene_panel_filename,
    SAGITTAL_DATA_ROOT / "Supp_table" / gene_panel_filename,
]
gene_panel_path = next((p for p in gene_panel_candidates if input_path(Path(p)).exists()), None)
if gene_panel_path is None:
    raise FileNotFoundError(
        "Could not find the MERFISH gene-panel table. Checked:\n"
        + "\n".join([str(p) for p in gene_panel_candidates])
    )

genepanel_table = pd.read_excel(input_path(gene_panel_path))
required_cols = {"Rationale for inclusion", "Vizgen Gene"}
missing_cols = required_cols.difference(genepanel_table.columns)
if len(missing_cols) > 0:
    raise KeyError(f"MERFISH gene-panel table is missing columns: {sorted(missing_cols)}")

rationale = genepanel_table["Rationale for inclusion"].fillna("").astype(str)
aging_mask = rationale.str.contains("aging|senescence", case=False, regex=True)

aging_genes = (
    genepanel_table.loc[aging_mask, "Vizgen Gene"]
    .dropna()
    .astype(str)
    .map(lambda x: x.strip())
)
aging_genes = [g for g in aging_genes if len(g) > 0]
aging_genes = list(dict.fromkeys(aging_genes))

if len(aging_genes) == 0:
    raise ValueError("No aging/senescence genes were found from the MERFISH gene-panel rationale column.")

# Resolve aging genes in the Coronal training expression matrix.
coronal_ref_var_names = processed_train.adata_list[0].var_names.astype(str)
coronal_resolved_genes, coronal_missing_aging_genes = _sagittal_resolve_gene_names_from_varnames(
    coronal_ref_var_names,
    aging_genes,
)
coronal_aging_genes_used_requested = [g for g in aging_genes if g in coronal_resolved_genes]
coronal_aging_genes_actual = [coronal_resolved_genes[g] for g in coronal_aging_genes_used_requested]

if len(coronal_aging_genes_actual) == 0:
    raise ValueError("None of the aging/senescence genes are present in the Coronal training expression matrix.")

print(
    "Coronal aging genes available for good-gene selection: "
    f"{len(coronal_aging_genes_actual)} / {len(aging_genes)}"
)
if len(coronal_missing_aging_genes) > 0:
    print(f"[Warning] Aging genes missing from Coronal training matrix skipped: {coronal_missing_aging_genes}")

# -----------------------------
# 1B. Split Coronal slices into 5 young + 5 old slices by age
# -----------------------------
coronal_slice_rows = []
for slice_index, adata_cur in enumerate(processed_train.adata_list):
    age_value, age_numeric = _sagittal_get_numeric_slice_age_from_adata(
        adata_cur,
        ["age", SAMPLE_ID_COL, "sample_age", "Age", "sample", "sample_name"],
    )
    coronal_slice_rows.append({
        "slice_index": int(slice_index),
        "age": age_value,
        "age_numeric": age_numeric,
        "n_cells": int(adata_cur.n_obs),
    })

coronal_slice_age_df = pd.DataFrame(coronal_slice_rows).sort_values(
    ["age_numeric", "slice_index"]
).reset_index(drop=True)

if coronal_slice_age_df["age_numeric"].isna().any():
    bad_slices = coronal_slice_age_df.loc[
        coronal_slice_age_df["age_numeric"].isna(),
        ["slice_index", "age"],
    ]
    raise ValueError(
        "Some Coronal slices have non-numeric age values and cannot be sorted into young/old groups:\n"
        + str(bad_slices)
    )

required_n_slices = CORONAL_GOOD_GENE_N_YOUNG_SLICES + CORONAL_GOOD_GENE_N_OLD_SLICES
if coronal_slice_age_df.shape[0] < required_n_slices:
    raise ValueError(
        f"Need at least {required_n_slices} Coronal slices to select "
        f"{CORONAL_GOOD_GENE_N_YOUNG_SLICES} young + {CORONAL_GOOD_GENE_N_OLD_SLICES} old slices, "
        f"but found only {coronal_slice_age_df.shape[0]} slices."
    )

young_coronal_slice_ids = (
    coronal_slice_age_df.head(CORONAL_GOOD_GENE_N_YOUNG_SLICES)["slice_index"]
    .astype(int)
    .tolist()
)
old_coronal_slice_ids = (
    coronal_slice_age_df.tail(CORONAL_GOOD_GENE_N_OLD_SLICES)["slice_index"]
    .astype(int)
    .tolist()
)
coronal_slice_group_map = {
    **{sid: "Young" for sid in young_coronal_slice_ids},
    **{sid: "Old" for sid in old_coronal_slice_ids},
}

coronal_slice_age_df["AgeGroup_5young_5old"] = coronal_slice_age_df["slice_index"].map(coronal_slice_group_map)
coronal_selected_slice_age_df = coronal_slice_age_df[
    coronal_slice_age_df["AgeGroup_5young_5old"].isin(["Young", "Old"])
].copy()

print("Coronal slices selected for good aging gene filtering:")
display(coronal_selected_slice_age_df)

# -----------------------------
# 1C. Compute old-vs-young log2FC for each aging gene in Coronal
# -----------------------------
coronal_slice_mean_rows = []
cell_sum_by_group = {
    "Young": np.zeros(len(coronal_aging_genes_actual), dtype=float),
    "Old": np.zeros(len(coronal_aging_genes_actual), dtype=float),
}
cell_count_by_group = {"Young": 0, "Old": 0}

for slice_index, adata_cur in enumerate(processed_train.adata_list):
    missing_cur = [g for g in coronal_aging_genes_actual if g not in adata_cur.var_names]
    if len(missing_cur) > 0:
        raise ValueError(
            f"Coronal slice {slice_index} is missing aging genes present in slice 0: {missing_cur}"
        )

    group_cur = coronal_slice_group_map.get(int(slice_index), None)
    if group_cur not in {"Young", "Old"}:
        continue

    X_gene_cur = _sagittal_get_dense_gene_matrix(adata_cur, coronal_aging_genes_actual)
    slice_gene_mean = np.nanmean(X_gene_cur, axis=0)

    age_row = coronal_slice_age_df.loc[coronal_slice_age_df["slice_index"] == int(slice_index)].iloc[0]
    for requested_gene, actual_gene, mean_expr in zip(
        coronal_aging_genes_used_requested,
        coronal_aging_genes_actual,
        slice_gene_mean,
    ):
        coronal_slice_mean_rows.append({
            "slice_index": int(slice_index),
            "age": age_row["age"],
            "age_numeric": float(age_row["age_numeric"]),
            "AgeGroup_5young_5old": group_cur,
            "n_cells": int(adata_cur.n_obs),
            "requested_gene": requested_gene,
            "actual_gene_coronal": actual_gene,
            "slice_mean_expression": float(mean_expr),
        })

    # Cell-level pooled mean expression by age group.
    cell_sum_by_group[group_cur] += np.nansum(X_gene_cur, axis=0)
    cell_count_by_group[group_cur] += int(X_gene_cur.shape[0])

coronal_aging_gene_slice_mean_df = pd.DataFrame(coronal_slice_mean_rows)

lfc_rows = []
for requested_gene, actual_gene in zip(coronal_aging_genes_used_requested, coronal_aging_genes_actual):
    df_gene = coronal_aging_gene_slice_mean_df[
        coronal_aging_gene_slice_mean_df["actual_gene_coronal"] == actual_gene
    ].copy()

    young_slice_vals = pd.to_numeric(
        df_gene.loc[df_gene["AgeGroup_5young_5old"] == "Young", "slice_mean_expression"],
        errors="coerce",
    ).dropna()
    old_slice_vals = pd.to_numeric(
        df_gene.loc[df_gene["AgeGroup_5young_5old"] == "Old", "slice_mean_expression"],
        errors="coerce",
    ).dropna()

    mean_young_slice = float(np.nanmean(young_slice_vals)) if len(young_slice_vals) > 0 else np.nan
    mean_old_slice = float(np.nanmean(old_slice_vals)) if len(old_slice_vals) > 0 else np.nan
    log2fc_slice_mean_old_vs_young = np.log2(
        (mean_old_slice + CORONAL_GOOD_GENE_LFC_PSEUDOCOUNT)
        / (mean_young_slice + CORONAL_GOOD_GENE_LFC_PSEUDOCOUNT)
    )

    gene_pos = coronal_aging_genes_actual.index(actual_gene)
    mean_young_cell = (
        float(cell_sum_by_group["Young"][gene_pos] / cell_count_by_group["Young"])
        if cell_count_by_group["Young"] > 0 else np.nan
    )
    mean_old_cell = (
        float(cell_sum_by_group["Old"][gene_pos] / cell_count_by_group["Old"])
        if cell_count_by_group["Old"] > 0 else np.nan
    )
    log2fc_cell_mean_old_vs_young = np.log2(
        (mean_old_cell + CORONAL_GOOD_GENE_LFC_PSEUDOCOUNT)
        / (mean_young_cell + CORONAL_GOOD_GENE_LFC_PSEUDOCOUNT)
    )

    lfc_rows.append({
        "requested_gene": requested_gene,
        "actual_gene_coronal": actual_gene,
        "n_young_slices": int(len(young_slice_vals)),
        "n_old_slices": int(len(old_slice_vals)),
        "mean_young_slice_mean_expression": mean_young_slice,
        "mean_old_slice_mean_expression": mean_old_slice,
        "log2FC_old_vs_young_slice_mean": float(log2fc_slice_mean_old_vs_young),
        "mean_young_cell_expression": mean_young_cell,
        "mean_old_cell_expression": mean_old_cell,
        "log2FC_old_vs_young_cell_mean": float(log2fc_cell_mean_old_vs_young),
        "old_greater_than_young_cell_mean": bool(log2fc_cell_mean_old_vs_young > 0),
        "good_aging_gene_cellLFC_gt_threshold": bool(
            log2fc_cell_mean_old_vs_young > GOOD_AGING_GENE_CELL_LFC_THRESHOLD
        ),
    })

coronal_aging_gene_old_vs_young_lfc_df = pd.DataFrame(lfc_rows).sort_values(
    "log2FC_old_vs_young_cell_mean",
    ascending=False,
).reset_index(drop=True)

# Keep this generic variable name for compatibility with the previous good-aging-gene workflow.
aging_gene_old_vs_young_lfc_df = coronal_aging_gene_old_vs_young_lfc_df.copy()

good_coronal_lfc_df = coronal_aging_gene_old_vs_young_lfc_df[
    np.isfinite(coronal_aging_gene_old_vs_young_lfc_df["log2FC_old_vs_young_cell_mean"])
    & (
        coronal_aging_gene_old_vs_young_lfc_df["log2FC_old_vs_young_cell_mean"]
        > GOOD_AGING_GENE_CELL_LFC_THRESHOLD
    )
].copy()

if good_coronal_lfc_df.shape[0] == 0:
    raise ValueError(
        "No Coronal aging genes passed the good-gene filter: "
        f"log2FC_old_vs_young_cell_mean > {GOOD_AGING_GENE_CELL_LFC_THRESHOLD}."
    )

print(
    "Coronal good aging genes selected: "
    f"{good_coronal_lfc_df.shape[0]} / {coronal_aging_gene_old_vs_young_lfc_df.shape[0]} "
    f"with log2FC_old_vs_young_cell_mean > {GOOD_AGING_GENE_CELL_LFC_THRESHOLD}"
)
display(coronal_aging_gene_old_vs_young_lfc_df)

# Save Coronal good-gene selection outputs into the Sagittal run directory
coronal_slice_path = sagittal_outdir / "Coronal_good_aging_gene_filter_selected_5young_5old_slices.csv"
coronal_lfc_path = sagittal_outdir / "Coronal_aging_gene_old_vs_young_log2FC_for_Sagittal_good_gene_filter.csv"
coronal_slice_mean_path = sagittal_outdir / "Coronal_aging_gene_slice_mean_expression_for_Sagittal_good_gene_filter.csv"
coronal_selected_slice_age_df.to_csv(output_path(coronal_slice_path), index=False)
coronal_aging_gene_old_vs_young_lfc_df.to_csv(output_path(coronal_lfc_path), index=False)
coronal_aging_gene_slice_mean_df.to_csv(output_path(coronal_slice_mean_path), index=False)
print(f"Saved Coronal selected slices to: {coronal_slice_path}")
print(f"Saved Coronal aging-gene log2FC table to: {coronal_lfc_path}")
print(f"Saved Coronal slice-mean expression table to: {coronal_slice_mean_path}")

# -----------------------------
# 1D. Resolve Coronal-selected good aging genes in the Sagittal matrix
# -----------------------------
good_requested_gene_set = set(good_coronal_lfc_df["requested_gene"].astype(str))

ref_var_names = processed_sagittal.adata_list[0].var_names.astype(str)
resolved_genes, missing_aging_genes = _sagittal_resolve_gene_names_from_varnames(
    ref_var_names,
    aging_genes,
)

aging_genes_used_requested = []
aging_genes_actual = []
for requested_gene in aging_genes:
    if requested_gene not in good_requested_gene_set:
        continue
    if requested_gene in resolved_genes:
        aging_genes_used_requested.append(requested_gene)
        aging_genes_actual.append(resolved_genes[requested_gene])

if len(aging_genes_actual) == 0:
    raise ValueError(
        "None of the Coronal-selected good aging genes are present in the Sagittal expression matrix."
    )

sagittal_good_gene_lfc_df = good_coronal_lfc_df[
    good_coronal_lfc_df["requested_gene"].isin(aging_genes_used_requested)
].copy()

print(
    "Sagittal aging genes used after Coronal good-gene filtering: "
    f"{len(aging_genes_actual)} / {len(good_requested_gene_set)} Coronal good genes"
)
if len(missing_aging_genes) > 0:
    print(f"[Warning] Aging genes missing from Sagittal matrix skipped: {missing_aging_genes}")

display(
    sagittal_good_gene_lfc_df[
        [
            "requested_gene",
            "actual_gene_coronal",
            "mean_young_cell_expression",
            "mean_old_cell_expression",
            "log2FC_old_vs_young_cell_mean",
            "log2FC_old_vs_young_slice_mean",
        ]
    ]
)

# -----------------------------
# 1E. Build Sagittal Aging module score with Coronal-selected good genes
# -----------------------------
X_aging_parts = []
barcode_parts = []
age_parts = []
age_numeric_parts = []
age_group_parts = []
celltype_parts = []
slice_parts = []
sample_id_parts = []

for slice_index, adata_cur in enumerate(processed_sagittal.adata_list):
    missing_cur = [g for g in aging_genes_actual if g not in adata_cur.var_names]
    if len(missing_cur) > 0:
        raise ValueError(
            f"Sagittal slice {slice_index} is missing Coronal-selected good aging genes: {missing_cur}"
        )

    X_aging_parts.append(_sagittal_get_dense_gene_matrix(adata_cur, aging_genes_actual))

    age_value = _sagittal_first_nonempty_obs_value(
        adata_cur,
        ["age", SAMPLE_ID_COL, "sample_age", "Age"],
        default=np.nan,
    )
    age_group, age_numeric = _sagittal_assign_age_group(
        age_value,
        old_age_threshold=globals().get("OLD_AGE_THRESHOLD", 19),
    )
    sample_id = _sagittal_first_nonempty_obs_value(
        adata_cur,
        ["sample", "sample_name_full", "sample_name", "SAMPLE_ID", SAMPLE_ID_COL, "age"],
        default=f"Sagittal_slice_{slice_index}",
    )

    barcode_parts.extend(adata_cur.obs_names.astype(str).tolist())
    slice_parts.extend([slice_index] * adata_cur.n_obs)
    sample_id_parts.extend([sample_id] * adata_cur.n_obs)
    age_parts.extend([age_value] * adata_cur.n_obs)
    age_numeric_parts.extend([age_numeric] * adata_cur.n_obs)
    age_group_parts.extend([age_group] * adata_cur.n_obs)

    if CELL_TYPE_COL in adata_cur.obs.columns:
        celltype_parts.extend(adata_cur.obs[CELL_TYPE_COL].astype(str).tolist())
    else:
        celltype_parts.extend([np.nan] * adata_cur.n_obs)

X_aging_all = np.vstack(X_aging_parts).astype(float, copy=False)

gene_mean = np.nanmean(X_aging_all, axis=0)
gene_std = np.nanstd(X_aging_all, axis=0, ddof=1)
gene_std[(~np.isfinite(gene_std)) | (gene_std == 0)] = np.nan

X_aging_global_z = (X_aging_all - gene_mean) / gene_std
X_aging_global_z = np.nan_to_num(X_aging_global_z, nan=0.0, posinf=0.0, neginf=0.0)

sagittal_aging_module_score = np.mean(X_aging_global_z, axis=1)
sagittal_aging_module_score = np.asarray(sagittal_aging_module_score, dtype=float)

sagittal_aging_module_score_df = pd.DataFrame({
    "barcode": barcode_parts,
    "slice_index": slice_parts,
    "sample_id": sample_id_parts,
    "age": age_parts,
    "age_numeric": age_numeric_parts,
    "AgeGroup": age_group_parts,
    "celltype": celltype_parts,
    "AgingGeneExp": sagittal_aging_module_score,
    "score_method": SAGITTAL_MODULE_SCORE_METHOD,
    "gene_filter_source": "Coronal",
    "gene_filter": f"Coronal log2FC_old_vs_young_cell_mean>{GOOD_AGING_GENE_CELL_LFC_THRESHOLD}",
})
sagittal_aging_module_score_df.index = barcode_parts

sagittal_aging_gene_used_df = pd.DataFrame({
    "requested_gene": aging_genes_used_requested,
    "actual_gene": aging_genes_actual,
    "gene_mean_global_sagittal": gene_mean,
    "gene_std_global_sagittal": gene_std,
})

sagittal_aging_gene_used_df = sagittal_aging_gene_used_df.merge(
    sagittal_good_gene_lfc_df[
        [
            "requested_gene",
            "actual_gene_coronal",
            "mean_young_cell_expression",
            "mean_old_cell_expression",
            "log2FC_old_vs_young_cell_mean",
            "mean_young_slice_mean_expression",
            "mean_old_slice_mean_expression",
            "log2FC_old_vs_young_slice_mean",
        ]
    ],
    on="requested_gene",
    how="left",
)

score_path = sagittal_outdir / "Sagittal_aging_module_score_CoronalGoodGenes_cellLFC_gt0p2_zscore_mean_global.csv"
gene_path = sagittal_outdir / "Sagittal_aging_module_score_CoronalGoodGenes_cellLFC_gt0p2_zscore_mean_global_genes_used.csv"
compat_score_path = sagittal_outdir / "Sagittal_aging_module_score_zscore_mean_global.csv"
compat_gene_path = sagittal_outdir / "Sagittal_aging_module_score_zscore_mean_global_genes_used.csv"

sagittal_aging_module_score_df.to_csv(output_path(score_path))
sagittal_aging_gene_used_df.to_csv(output_path(gene_path), index=False)

# Compatibility outputs used by older downstream code / file checks.
sagittal_aging_module_score_df.to_csv(output_path(compat_score_path))
sagittal_aging_gene_used_df.to_csv(output_path(compat_gene_path), index=False)

print(f"Saved Sagittal Aging module scores using Coronal-selected good genes to: {score_path}")
print(f"Saved Sagittal good aging genes used to: {gene_path}")
display(sagittal_aging_module_score_df.head())
display(sagittal_aging_gene_used_df)

# -----------------------------
# 2. T cell outgoing MI-29 grouping and Aging module score comparison
# -----------------------------
tcell_rows = []
start_index = 0

for batch_index, (adata_cur, data_cur, factor_cur) in enumerate(
    zip(
        processed_sagittal.adata_list,
        processed_sagittal.spidernet_data,
        sagittal_transfer_results["factor_envir_list"],
    )
):
    edge_index_cur = _sagittal_get_edge_index_numpy(data_cur)
    factor_envir_cur = _sagittal_get_factor_numpy(factor_cur)

    if SAGITTAL_MI_INDEX >= factor_envir_cur.shape[1]:
        raise IndexError(
            f"{SAGITTAL_MI_OI} requires column {SAGITTAL_MI_INDEX}, "
            f"but factor_envir has only {factor_envir_cur.shape[1]} columns."
        )

    end_index = start_index + adata_cur.n_obs
    aging_score_cur = sagittal_aging_module_score[start_index:end_index]
    start_index = end_index

    celltype_all_cur = np.asarray(adata_cur.obs[CELL_TYPE_COL].astype(str))
    sender_is_tcell = celltype_all_cur[edge_index_cur[:, 0]] == SAGITTAL_SENDER_CELLTYPE

    if np.sum(sender_is_tcell) == 0:
        continue

    edge_df = pd.DataFrame({
        "sender_index": edge_index_cur[sender_is_tcell, 0].astype(int),
        "receiver_index": edge_index_cur[sender_is_tcell, 1].astype(int),
        "MI29_strength": factor_envir_cur[sender_is_tcell, SAGITTAL_MI_INDEX].astype(float),
    })

    sender_agg = edge_df.groupby("sender_index", sort=False).agg(
        Tcell_to_neighbor_MI29_sum=("MI29_strength", "sum"),
        Tcell_to_neighbor_MI29_mean=("MI29_strength", "mean"),
        Tcell_to_neighbor_MI29_max=("MI29_strength", "max"),
        n_neighbor_edges=("MI29_strength", "size"),
    )

    age_value = _sagittal_first_nonempty_obs_value(
        adata_cur,
        ["age", SAMPLE_ID_COL, "sample_age", "Age"],
        default=np.nan,
    )
    age_group, age_numeric = _sagittal_assign_age_group(
        age_value,
        old_age_threshold=globals().get("OLD_AGE_THRESHOLD", 19),
    )
    sample_id = _sagittal_first_nonempty_obs_value(
        adata_cur,
        ["sample", "sample_name_full", "sample_name", "SAMPLE_ID", SAMPLE_ID_COL, "age"],
        default=f"Sagittal_slice_{batch_index}",
    )

    for sender_index, row in sender_agg.iterrows():
        receiver_indices = edge_df.loc[
            edge_df["sender_index"] == sender_index,
            "receiver_index",
        ].to_numpy(dtype=int)

        receiver_indices = receiver_indices[
            (receiver_indices >= 0) & (receiver_indices < len(aging_score_cur))
        ]

        neighbor_score = (
            float(np.nanmean(aging_score_cur[receiver_indices]))
            if len(receiver_indices) > 0
            else np.nan
        )

        tcell_rows.append({
            "Dataset": "Sagittal",
            "slice_index": batch_index,
            "sample_id": sample_id,
            "age": age_value,
            "age_numeric": age_numeric,
            "AgeGroup": age_group,
            "sender_celltype": SAGITTAL_SENDER_CELLTYPE,
            "sender_index": int(sender_index),
            "sender_barcode": str(adata_cur.obs_names[int(sender_index)]),
            "Tcell_to_neighbor_MI29_sum": float(row["Tcell_to_neighbor_MI29_sum"]),
            "Tcell_to_neighbor_MI29_mean": float(row["Tcell_to_neighbor_MI29_mean"]),
            "Tcell_to_neighbor_MI29_max": float(row["Tcell_to_neighbor_MI29_max"]),
            "n_neighbor_edges": int(row["n_neighbor_edges"]),
            "AgingGeneExp": float(aging_score_cur[int(sender_index)]),
            "AgingGeneExp_neighbor": neighbor_score,
        })

sagittal_tcell_mi29_aging_df = pd.DataFrame(tcell_rows)

# Use summed outgoing T cell -> neighbor MI-29 for High/Low grouping.
# This matches AgingBrain_training_analysis, where mean_values = sum_values before qcut grouping.
sagittal_tcell_mi29_aging_df = sagittal_tcell_mi29_aging_df[
    np.isfinite(sagittal_tcell_mi29_aging_df["Tcell_to_neighbor_MI29_sum"])
    & (sagittal_tcell_mi29_aging_df["Tcell_to_neighbor_MI29_sum"] > 0)
].copy()

sagittal_tcell_mi29_aging_df = _sagittal_assign_high_low_by_quantile(
    sagittal_tcell_mi29_aging_df,
    value_col="Tcell_to_neighbor_MI29_sum",
    group_col="Tcell_MI29_group",
    q=2,
)

grouped_data_path = sagittal_outdir / "Sagittal_Tcell_MI29_sum_group_Aging_module_score_table.csv"
sagittal_tcell_mi29_aging_df.to_csv(output_path(grouped_data_path), index=False)
print(f"Saved Sagittal T-cell MI-29 grouping table to: {grouped_data_path}")

display(
    sagittal_tcell_mi29_aging_df[
        [
            "Dataset",
            "slice_index",
            "sample_id",
            "AgeGroup",
            "sender_barcode",
            "Tcell_to_neighbor_MI29_sum",
            "Tcell_to_neighbor_MI29_mean",
            "Tcell_to_neighbor_MI29_max",
            "Tcell_MI29_group",
            "AgingGeneExp",
            "AgingGeneExp_neighbor",
        ]
    ].head()
)
sagittal_tcell_aging_stats = _sagittal_plot_grouped_aging_score(
    data=sagittal_tcell_mi29_aging_df,
    score_col="AgingGeneExp",
    output_prefix="Sagittal_AgingGeneExp_Tcell_by_TcellSendingMI29_SumGroup_Boxplot",
    y_label="T-cell aging module score",
    x_label="T cell group\n(summed outgoing MI-29)",
    color_map=SAGITTAL_TCELL_GROUP_PALETTE,
)

sagittal_neighbor_aging_stats = _sagittal_plot_grouped_aging_score(
    data=sagittal_tcell_mi29_aging_df,
    score_col="AgingGeneExp_neighbor",
    output_prefix="Sagittal_AgingGeneExp_neighbor_by_TcellSendingMI29_SumGroup_Boxplot",
    y_label="Neighboring-cell aging module score",
    x_label="Neighbors of T cells\n(by summed outgoing MI-29 group)",
    color_map=SAGITTAL_NEIGHBOR_GROUP_PALETTE,
)

sagittal_tcell_and_neighbor_aging_stats = pd.concat(
    [sagittal_tcell_aging_stats, sagittal_neighbor_aging_stats],
    axis=0,
    ignore_index=True,
)

combined_stats_path = sagittal_outdir / "Sagittal_AgingGeneExp_Tcell_and_neighbor_by_TcellSendingMI29_SumGroup_stats.csv"
sagittal_tcell_and_neighbor_aging_stats.to_csv(output_path(combined_stats_path), index=False)
print(f"Saved combined Sagittal Aging module score comparison stats to: {combined_stats_path}")

In [ ]:
# ==============================================================
# Receiver-cell-type-specific neighboring-cell ageing score
# --------------------------------------------------------------
# Separate neighboring receiver cells by cell type. For each T cell and
# receiver cell type, compute the mean ageing-module score of those neighbors,
# then compare the MI-29-high and MI-29-low T-cell groups.
#
# Required upstream objects from the preceding sagittal ageing-score analysis:
#   processed_sagittal
#   sagittal_transfer_results
#   sagittal_aging_module_score
#   sagittal_tcell_mi29_aging_df
#   sagittal_outdir
#   _sagittal_get_edge_index_numpy
#   _sagittal_compute_smd
#   _sagittal_p_label
# ==============================================================
sagittal_outdir = Path(sagittal_run_dirs["run_dir"])
sagittal_outdir.mkdir(parents=True, exist_ok=True)

from pathlib import Path
import re
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu


# -----------------------------
# Settings
# -----------------------------
SAGITTAL_RECEIVER_TYPE_MIN_TCELLS_PER_GROUP = 3
SAGITTAL_RECEIVER_TYPE_MAX_PANELS = None  # set to an integer if you want only top N receiver types
SAGITTAL_RECEIVER_TYPE_POINT_SIZE = 1.7
SAGITTAL_RECEIVER_TYPE_POINT_ALPHA = 0.28

receiver_type_outdir = sagittal_outdir / "receiver_celltype_specific_neighbor_aging"
receiver_type_outdir.mkdir(parents=True, exist_ok=True)


def _safe_filename(x):
    x = str(x)
    x = re.sub(r"[^\w\-.]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x if len(x) > 0 else "NA"


def _plot_one_receiver_type_boxplot(
    plot_df,
    receiver_celltype,
    stats_row,
    out_prefix,
):
    plt.figure(figsize=(3.2, 4.0))
    ax = plt.gca()

    sns.boxplot(
        data=plot_df,
        x="Tcell_MI29_group",
        y="AgingGeneExp_neighbor_receiver_type_mean",
        order=SAGITTAL_TCELL_MI29_GROUP_ORDER,
        palette=SAGITTAL_NEIGHBOR_GROUP_PALETTE,
        width=0.7,
        linewidth=1.4,
        showfliers=False,
        medianprops=dict(color="white", linewidth=1.8),
        capprops=dict(linewidth=0),
        ax=ax,
    )

    sns.stripplot(
        data=plot_df,
        x="Tcell_MI29_group",
        y="AgingGeneExp_neighbor_receiver_type_mean",
        order=SAGITTAL_TCELL_MI29_GROUP_ORDER,
        color="black",
        size=SAGITTAL_RECEIVER_TYPE_POINT_SIZE,
        jitter=0.18,
        alpha=SAGITTAL_RECEIVER_TYPE_POINT_ALPHA,
        ax=ax,
    )

    pval = stats_row["pvalue"]
    smd = stats_row["smd_high_minus_low"]

    ax.text(
        0.5,
        0.98,
        f"{_sagittal_p_label(pval)}\nSMD = {smd:.2f}",
        ha="center",
        va="top",
        fontsize=10.5,
        transform=ax.transAxes,
    )

    ax.set_title(str(receiver_celltype), fontsize=13)
    ax.set_xlabel("T cell group\n(summed outgoing MI-29)", fontsize=12)
    ax.set_ylabel("Neighboring-cell aging module score", fontsize=12)
    ax.tick_params(axis="both", labelsize=11)

    sns.despine(top=True, right=True)
    plt.tight_layout()

    plot_path = receiver_type_outdir / f"{out_prefix}.pdf"
    plt.savefig(output_path(plot_path), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    return plot_path


if "sagittal_tcell_mi29_aging_df" not in globals():
    raise NameError("Please run Cell 2 first so that sagittal_tcell_mi29_aging_df exists.")

if "sagittal_aging_module_score" not in globals():
    raise NameError("Please run Cell 2 first so that sagittal_aging_module_score exists.")

required_cols = {
    "slice_index",
    "sender_index",
    "sender_barcode",
    "Tcell_MI29_group",
    "Tcell_to_neighbor_MI29_sum",
}
missing_cols = required_cols.difference(sagittal_tcell_mi29_aging_df.columns)
if len(missing_cols) > 0:
    raise KeyError(f"sagittal_tcell_mi29_aging_df is missing columns: {sorted(missing_cols)}")


# -----------------------------
# 1. Build edge-level table:
#    each row = one T cell -> receiver-cell edge
# -----------------------------
tcell_group_df = sagittal_tcell_mi29_aging_df.copy()
tcell_group_df = tcell_group_df[
    tcell_group_df["Tcell_MI29_group"].isin(SAGITTAL_TCELL_MI29_GROUP_ORDER)
].copy()

tcell_group_df["slice_index"] = tcell_group_df["slice_index"].astype(int)
tcell_group_df["sender_index"] = tcell_group_df["sender_index"].astype(int)
tcell_group_df["Tcell_MI29_group"] = tcell_group_df["Tcell_MI29_group"].astype(str)

edge_receiver_rows = []
start_index = 0

for batch_index, (adata_cur, data_cur, factor_cur) in enumerate(
    zip(
        processed_sagittal.adata_list,
        processed_sagittal.spidernet_data,
        sagittal_transfer_results["factor_envir_list"],
    )
):
    edge_index_cur = _sagittal_get_edge_index_numpy(data_cur)

    end_index = start_index + adata_cur.n_obs
    aging_score_cur = np.asarray(sagittal_aging_module_score[start_index:end_index], dtype=float)
    start_index = end_index

    celltype_all_cur = np.asarray(adata_cur.obs[CELL_TYPE_COL].astype(str))
    obs_names_cur = np.asarray(adata_cur.obs_names.astype(str))

    # Only T cell sender edges
    sender_is_tcell = celltype_all_cur[edge_index_cur[:, 0]] == SAGITTAL_SENDER_CELLTYPE
    if np.sum(sender_is_tcell) == 0:
        continue

    edge_tcell_cur = pd.DataFrame({
        "sender_index": edge_index_cur[sender_is_tcell, 0].astype(int),
        "receiver_index": edge_index_cur[sender_is_tcell, 1].astype(int),
    })

    # Keep only T cells that were assigned High/Low groups in Cell 2
    group_cur = tcell_group_df.loc[
        tcell_group_df["slice_index"] == batch_index,
        [
            "sender_index",
            "sender_barcode",
            "sample_id",
            "AgeGroup",
            "Tcell_MI29_group",
            "Tcell_to_neighbor_MI29_sum",
            "Tcell_to_neighbor_MI29_mean",
            "Tcell_to_neighbor_MI29_max",
            "n_neighbor_edges",
        ],
    ].copy()

    if group_cur.shape[0] == 0:
        continue

    group_cur = group_cur.drop_duplicates(subset=["sender_index"]).set_index("sender_index")
    edge_tcell_cur = edge_tcell_cur[
        edge_tcell_cur["sender_index"].isin(group_cur.index)
    ].copy()

    if edge_tcell_cur.shape[0] == 0:
        continue

    # Valid receiver indices
    valid_receiver = (
        (edge_tcell_cur["receiver_index"].to_numpy() >= 0)
        & (edge_tcell_cur["receiver_index"].to_numpy() < adata_cur.n_obs)
    )
    edge_tcell_cur = edge_tcell_cur.loc[valid_receiver].copy()

    if edge_tcell_cur.shape[0] == 0:
        continue

    edge_tcell_cur["Dataset"] = "Sagittal"
    edge_tcell_cur["slice_index"] = batch_index

    # Sender-side group metadata
    edge_tcell_cur["sender_barcode"] = edge_tcell_cur["sender_index"].map(group_cur["sender_barcode"])
    edge_tcell_cur["sample_id"] = edge_tcell_cur["sender_index"].map(group_cur["sample_id"])
    edge_tcell_cur["AgeGroup"] = edge_tcell_cur["sender_index"].map(group_cur["AgeGroup"])
    edge_tcell_cur["Tcell_MI29_group"] = edge_tcell_cur["sender_index"].map(group_cur["Tcell_MI29_group"])
    edge_tcell_cur["Tcell_to_neighbor_MI29_sum"] = edge_tcell_cur["sender_index"].map(group_cur["Tcell_to_neighbor_MI29_sum"])
    edge_tcell_cur["Tcell_to_neighbor_MI29_mean"] = edge_tcell_cur["sender_index"].map(group_cur["Tcell_to_neighbor_MI29_mean"])
    edge_tcell_cur["Tcell_to_neighbor_MI29_max"] = edge_tcell_cur["sender_index"].map(group_cur["Tcell_to_neighbor_MI29_max"])
    edge_tcell_cur["n_neighbor_edges_sender_total"] = edge_tcell_cur["sender_index"].map(group_cur["n_neighbor_edges"])

    # Receiver-side metadata
    recv_idx = edge_tcell_cur["receiver_index"].to_numpy(dtype=int)
    edge_tcell_cur["receiver_barcode"] = obs_names_cur[recv_idx]
    edge_tcell_cur["receiver_celltype"] = celltype_all_cur[recv_idx]
    edge_tcell_cur["AgingGeneExp_receiver"] = aging_score_cur[recv_idx]

    edge_receiver_rows.append(edge_tcell_cur)


if len(edge_receiver_rows) == 0:
    raise ValueError("No T cell -> receiver-cell edges were found for receiver-cell-type analysis.")

sagittal_neighbor_receiver_type_edge_df = pd.concat(edge_receiver_rows, axis=0, ignore_index=True)
sagittal_neighbor_receiver_type_edge_df = sagittal_neighbor_receiver_type_edge_df[
    sagittal_neighbor_receiver_type_edge_df["Tcell_MI29_group"].isin(SAGITTAL_TCELL_MI29_GROUP_ORDER)
    & np.isfinite(pd.to_numeric(sagittal_neighbor_receiver_type_edge_df["AgingGeneExp_receiver"], errors="coerce"))
].copy()

edge_table_path = receiver_type_outdir / "Sagittal_Tcell_MI29_group_receiver_celltype_neighbor_aging_edge_table.csv"
sagittal_neighbor_receiver_type_edge_df.to_csv(output_path(edge_table_path), index=False)
print(f"Saved edge-level receiver-cell-type table to: {edge_table_path}")


# -----------------------------
# 2. Aggregate to one point per T cell x receiver cell type
#    This matches the original second boxplot logic:
#    neighboring receiver Aging score is averaged per T cell,
#    but now separately within each receiver cell type.
# -----------------------------
sagittal_neighbor_receiver_type_df = (
    sagittal_neighbor_receiver_type_edge_df
    .groupby(
        [
            "Dataset",
            "slice_index",
            "sample_id",
            "AgeGroup",
            "sender_index",
            "sender_barcode",
            "Tcell_MI29_group",
            "receiver_celltype",
        ],
        observed=True,
        sort=False,
    )
    .agg(
        AgingGeneExp_neighbor_receiver_type_mean=("AgingGeneExp_receiver", "mean"),
        AgingGeneExp_neighbor_receiver_type_median=("AgingGeneExp_receiver", "median"),
        n_receiver_edges_this_type=("AgingGeneExp_receiver", "size"),
        n_unique_receivers_this_type=("receiver_barcode", "nunique"),
        Tcell_to_neighbor_MI29_sum=("Tcell_to_neighbor_MI29_sum", "first"),
        Tcell_to_neighbor_MI29_mean=("Tcell_to_neighbor_MI29_mean", "first"),
        Tcell_to_neighbor_MI29_max=("Tcell_to_neighbor_MI29_max", "first"),
        n_neighbor_edges_sender_total=("n_neighbor_edges_sender_total", "first"),
    )
    .reset_index()
)

receiver_type_table_path = receiver_type_outdir / "Sagittal_Tcell_MI29_group_receiver_celltype_neighbor_aging_TcellMean_table.csv"
sagittal_neighbor_receiver_type_df.to_csv(output_path(receiver_type_table_path), index=False)
print(f"Saved T-cell x receiver-cell-type mean table to: {receiver_type_table_path}")

display(sagittal_neighbor_receiver_type_df.head())


# -----------------------------
# 3. Statistics for each receiver cell type
# -----------------------------
receiver_type_stats_rows = []

for receiver_celltype, df_ct in sagittal_neighbor_receiver_type_df.groupby("receiver_celltype", sort=False):
    df_ct = df_ct[
        df_ct["Tcell_MI29_group"].isin(SAGITTAL_TCELL_MI29_GROUP_ORDER)
        & np.isfinite(pd.to_numeric(df_ct["AgingGeneExp_neighbor_receiver_type_mean"], errors="coerce"))
    ].copy()

    high_vals = pd.to_numeric(
        df_ct.loc[df_ct["Tcell_MI29_group"] == "High", "AgingGeneExp_neighbor_receiver_type_mean"],
        errors="coerce",
    ).dropna()

    low_vals = pd.to_numeric(
        df_ct.loc[df_ct["Tcell_MI29_group"] == "Low", "AgingGeneExp_neighbor_receiver_type_mean"],
        errors="coerce",
    ).dropna()

    if len(high_vals) > 0 and len(low_vals) > 0:
        stat, pval = mannwhitneyu(low_vals, high_vals, alternative="two-sided")
    else:
        stat, pval = np.nan, np.nan

    smd = _sagittal_compute_smd(high_vals, low_vals)

    receiver_type_stats_rows.append({
        "Dataset": "Sagittal",
        "MI": SAGITTAL_MI_OI,
        "score": "AgingGeneExp_neighbor_receiver_type_mean",
        "receiver_celltype": receiver_celltype,
        "group_high": "High",
        "group_low": "Low",
        "n_high_tcells": int(len(high_vals)),
        "n_low_tcells": int(len(low_vals)),
        "mean_high": float(np.nanmean(high_vals)) if len(high_vals) > 0 else np.nan,
        "mean_low": float(np.nanmean(low_vals)) if len(low_vals) > 0 else np.nan,
        "smd_high_minus_low": smd,
        "mannwhitneyu_stat": stat,
        "pvalue": pval,
    })

sagittal_neighbor_receiver_type_stats = pd.DataFrame(receiver_type_stats_rows)

# Keep receiver types with enough T cells in both groups for plotting
plot_receiver_types = sagittal_neighbor_receiver_type_stats[
    (sagittal_neighbor_receiver_type_stats["n_high_tcells"] >= SAGITTAL_RECEIVER_TYPE_MIN_TCELLS_PER_GROUP)
    & (sagittal_neighbor_receiver_type_stats["n_low_tcells"] >= SAGITTAL_RECEIVER_TYPE_MIN_TCELLS_PER_GROUP)
].copy()

plot_receiver_types = plot_receiver_types.sort_values(
    ["smd_high_minus_low", "receiver_celltype"],
    ascending=[False, True],
)

if SAGITTAL_RECEIVER_TYPE_MAX_PANELS is not None:
    plot_receiver_types = plot_receiver_types.head(int(SAGITTAL_RECEIVER_TYPE_MAX_PANELS)).copy()

receiver_type_order = plot_receiver_types["receiver_celltype"].astype(str).tolist()

stats_path = receiver_type_outdir / "Sagittal_AgingGeneExp_neighbor_by_TcellSendingMI29_SumGroup_receiver_celltype_stats.csv"
sagittal_neighbor_receiver_type_stats.to_csv(output_path(stats_path), index=False)
print(f"Saved receiver-cell-type stats to: {stats_path}")

display(sagittal_neighbor_receiver_type_stats.sort_values("smd_high_minus_low", ascending=False))


# -----------------------------
# 4. Combined multi-panel plot
# -----------------------------
if len(receiver_type_order) == 0:
    print(
        "[Warning] No receiver cell type has enough T cells in both High and Low groups "
        f"with min per group = {SAGITTAL_RECEIVER_TYPE_MIN_TCELLS_PER_GROUP}."
    )
else:
    plot_df = sagittal_neighbor_receiver_type_df[
        sagittal_neighbor_receiver_type_df["receiver_celltype"].astype(str).isin(receiver_type_order)
        & sagittal_neighbor_receiver_type_df["Tcell_MI29_group"].isin(SAGITTAL_TCELL_MI29_GROUP_ORDER)
        & np.isfinite(pd.to_numeric(
            sagittal_neighbor_receiver_type_df["AgingGeneExp_neighbor_receiver_type_mean"],
            errors="coerce",
        ))
    ].copy()

    n_panels = len(receiver_type_order)
    n_cols = min(4, n_panels)
    n_rows = int(math.ceil(n_panels / n_cols))

    fig_width = max(3.2 * n_cols, 4.0)
    fig_height = max(3.7 * n_rows, 3.8)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(fig_width, fig_height),
        squeeze=False,
        sharey=False,
    )

    axes_flat = axes.ravel()

    stats_lookup = plot_receiver_types.set_index("receiver_celltype")

    for ax_i, receiver_celltype in enumerate(receiver_type_order):
        ax = axes_flat[ax_i]
        df_ct = plot_df[plot_df["receiver_celltype"].astype(str) == str(receiver_celltype)].copy()

        sns.boxplot(
            data=df_ct,
            x="Tcell_MI29_group",
            y="AgingGeneExp_neighbor_receiver_type_mean",
            order=SAGITTAL_TCELL_MI29_GROUP_ORDER,
            palette=SAGITTAL_NEIGHBOR_GROUP_PALETTE,
            width=0.7,
            linewidth=1.2,
            showfliers=False,
            medianprops=dict(color="white", linewidth=1.6),
            capprops=dict(linewidth=0),
            ax=ax,
        )

        sns.stripplot(
            data=df_ct,
            x="Tcell_MI29_group",
            y="AgingGeneExp_neighbor_receiver_type_mean",
            order=SAGITTAL_TCELL_MI29_GROUP_ORDER,
            color="black",
            size=SAGITTAL_RECEIVER_TYPE_POINT_SIZE,
            jitter=0.18,
            alpha=SAGITTAL_RECEIVER_TYPE_POINT_ALPHA,
            ax=ax,
        )

        stat_row = stats_lookup.loc[receiver_celltype]
        pval = stat_row["pvalue"]
        smd = stat_row["smd_high_minus_low"]

        ax.text(
            0.5,
            0.98,
            f"{_sagittal_p_label(pval)}\nSMD = {smd:.2f}",
            ha="center",
            va="top",
            fontsize=9.5,
            transform=ax.transAxes,
        )

        ax.set_title(str(receiver_celltype), fontsize=12)
        ax.set_xlabel("T cell group\n(summed MI-29)", fontsize=10.5)
        ax.set_ylabel("Neighbor aging score", fontsize=10.5)
        ax.tick_params(axis="both", labelsize=9.5)
        sns.despine(ax=ax, top=True, right=True)

    for j in range(n_panels, len(axes_flat)):
        axes_flat[j].axis("off")

    plt.tight_layout()

    combined_plot_path = receiver_type_outdir / "Sagittal_AgingGeneExp_neighbor_by_TcellSendingMI29_SumGroup_receiver_celltype_boxplots.pdf"
    plt.savefig(output_path(combined_plot_path), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    print(f"Saved combined receiver-cell-type boxplots to: {combined_plot_path}")


# -----------------------------
# 5. Also save one separate PDF per receiver cell type
# -----------------------------
separate_plot_paths = []

for receiver_celltype in receiver_type_order:
    df_ct = sagittal_neighbor_receiver_type_df[
        (sagittal_neighbor_receiver_type_df["receiver_celltype"].astype(str) == str(receiver_celltype))
        & sagittal_neighbor_receiver_type_df["Tcell_MI29_group"].isin(SAGITTAL_TCELL_MI29_GROUP_ORDER)
        & np.isfinite(pd.to_numeric(
            sagittal_neighbor_receiver_type_df["AgingGeneExp_neighbor_receiver_type_mean"],
            errors="coerce",
        ))
    ].copy()

    if df_ct.shape[0] == 0:
        continue

    stat_row = plot_receiver_types.set_index("receiver_celltype").loc[receiver_celltype]
    out_prefix = (
        "Sagittal_AgingGeneExp_neighbor_by_TcellSendingMI29_SumGroup_"
        f"receiver_{_safe_filename(receiver_celltype)}"
    )

    plot_path = _plot_one_receiver_type_boxplot(
        plot_df=df_ct,
        receiver_celltype=receiver_celltype,
        stats_row=stat_row,
        out_prefix=out_prefix,
    )
    separate_plot_paths.append(plot_path)

print(f"Saved {len(separate_plot_paths)} separate receiver-cell-type boxplots to: {receiver_type_outdir}")